# Hybrid feature-set builder with HIE stability and correlation audit

Этот notebook строит **правильный train-only pipeline** для проверки стабильности HIE-признаков и создания hybrid-наборов:

1. Загружает OHLCV через `KlinesDataLoader`.
2. Для каждого `z_window` строит market features и train/val/test split.
3. Для `alpha_out=0.5` считает HIE entry/exit на `full_train`, `early_train`, `mid_train`, `late_train`.
4. Объединяет entry/exit через raw unpruned union ranking.
5. Считает HIE stability **per z-window** и **global across z-windows**.
6. Считает Spearman-correlation ranking прямо внутри notebook на train.
7. Строит `corrPrunedTopK` baseline.
8. Строит hybrid set: `CorrCore + StableHIE incremental + fallback`, с финальным Spearman pruning.
9. Сохраняет audit: raw corr rank HIE-признаков, выкидывает ли corr pruning HIE-признаки, Jaccard hybrid vs corr. Raw corr top-N flags не используются; сохраняется именно raw rank.
10. Считает демонстрационный overlap HIE full-train `alpha=0.5` vs `alpha=1.0`.
11. В конце есть блок для перестройки hybrid sizes без пересчёта HIE.

Ключевой принцип:

> HIE stability считается **до Spearman-pruning**, потому что pruning — это redundancy control для финальной модели, а не критерий HIE-важности.


## 0. Imports, paths, and project setup

In [1]:
from pathlib import Path
import sys
import json
import warnings
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =====================================================================
# Paths: поменяй только PROJECT_ROOT при необходимости.
# =====================================================================
PROJECT_ROOT = Path(r"D:\PythonProjects\VKR")

# KlinesDataLoader.load_data ожидает относительный путь от project root внутри loader.
OHLCV_RELATIVE_PATH = r"data\klines_data\crypto_240m_bybit_TEST.parquet"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "feature_selection"
    / "hybrid_stable_hie_corr_outputs_alpha0p5_train_only_v2"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Add project folders to sys.path.
for p in [
    PROJECT_ROOT,
    PROJECT_ROOT / "data_processing",
    PROJECT_ROOT / "feature_selection",
    PROJECT_ROOT / "backtest",
]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OHLCV_RELATIVE_PATH:", OHLCV_RELATIVE_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


PROJECT_ROOT: D:\PythonProjects\VKR
OHLCV_RELATIVE_PATH: data\klines_data\crypto_240m_bybit_TEST.parquet
OUTPUT_DIR: D:\PythonProjects\VKR\feature_selection\hybrid_stable_hie_corr_outputs_alpha0p5_train_only_v2


## 1. Configuration

Основная семантика HIE: `alpha_out=0.5`. Она интерпретируется как **partial opportunity cost** для long/flat стратегии: flat — не short, поэтому missed upside штрафуется частично.

`alpha_out=1.0` считается только на `full_train` и только для демонстрационного overlap с `alpha=0.5`.


In [2]:
SEED = 42
ALL_SYMBOLS = ["BTCUSDT", "DOGEUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT"]

INTERVAL = 240
HORIZON = 10
TRADE_COST = 0.0025

# Main HIE semantics for stability/hybrid.
MAIN_HIE_ALPHA_OUT = 0.5

# Demonstration only: compute full-train HIE for alpha=1.0 on selected z_windows and compare with alpha=0.5.
ALPHA_OVERLAP_REFERENCE = 1.0
ALPHA_OVERLAP_DEMO_Z_WINDOWS = [72]  # можно поставить [24, 48, 72, 96], но это дольше.

VAL_BARS = 2000
TEST_BARS = 2000
EMBARGO_BARS = 0

Z_WINDOWS = [24, 48, 72, 96]
TRAIN_BLOCKS = ["early_train", "mid_train", "late_train"]

# HIE settings.
N_BINS = 15
N_BOOTSTRAP = 100
MIN_BIN_SIZE = 40
MIN_ACTION_COUNT_PER_BIN = 20

# HIE union/stability settings.
# Stability is evaluated on a sensitivity grid.
# Main K=14 is tied to the current hybrid design: about 6-7 HIE slots,
# with a 2x candidate buffer. K=26 is kept only as a loose robustness check.
# The entry/exit union ranking uses this main K for regime-coverage tie-breaking.
HIE_TOPK_STABILITY_GRID = [7, 14, 20, 26]
MAIN_HIE_STABILITY_TOPK = 14  # main raw unpruned union pool used for candidate stability
HIE_MIN_STABILITY_COUNT = 2   # appears in at least 2 of 3 train blocks

# Corr / hybrid settings.
PRUNE_THRESHOLD = 0.85
CORR_METHOD = "spearman"
DEFAULT_FINAL_K = 10
DEFAULT_CORR_CORE_K = 5
DEFAULT_HIE_TARGET_K = 5

# Audit: for each stable HIE feature we keep its raw Spearman correlation rank.
# Main hybrid does NOT exclude raw corr top-N by default.
STRICT_EXCLUDE_RAW_CORR_TOP_N_FOR_INCREMENTAL = None
# Optional strict robustness: set to an integer, e.g. 15, to exclude HIE candidates with corr_rank <= 15.

META_COLS = ["timestamp", "symbol", "open", "high", "low", "close", "volume"]

CONFIG_FOR_INDICATORS = {
    "ema_periods": [9, 21, 50, 100, 200],
    "momentum_indicators_periods": [14, 30, 72],
    "return_indicators_periods": [6, 24, 72, 168],
    "volatility_indicators_periods": [24, 72, 168],
    "level_periods": [24, 72, 168],
    "vol_ma_period": [24, 72, 168],
    "range_ma_period": [24, 72, 168],
}

config_for_audit = {
    "created_at": datetime.now().isoformat(),
    "loader": "KlinesDataLoader.load_data",
    "ohlcv_relative_path": OHLCV_RELATIVE_PATH,
    "main_hie_alpha_out": MAIN_HIE_ALPHA_OUT,
    "alpha_overlap_reference": ALPHA_OVERLAP_REFERENCE,
    "alpha_overlap_demo_z_windows": ALPHA_OVERLAP_DEMO_Z_WINDOWS,
    "horizon": HORIZON,
    "trade_cost": TRADE_COST,
    "z_windows": Z_WINDOWS,
    "train_blocks": TRAIN_BLOCKS,
    "hie_topk_stability_grid": HIE_TOPK_STABILITY_GRID,
    "main_hie_stability_topk": MAIN_HIE_STABILITY_TOPK,
    "hie_min_stability_count": HIE_MIN_STABILITY_COUNT,
    "n_bins": N_BINS,
    "n_bootstrap": N_BOOTSTRAP,
    "min_bin_size": MIN_BIN_SIZE,
    "min_action_count_per_bin": MIN_ACTION_COUNT_PER_BIN,
    "prune_threshold": PRUNE_THRESHOLD,
    "default_final_k": DEFAULT_FINAL_K,
    "default_corr_core_k": DEFAULT_CORR_CORE_K,
    "default_hie_target_k": DEFAULT_HIE_TARGET_K,
    "strict_exclude_raw_corr_top_n_for_incremental": STRICT_EXCLUDE_RAW_CORR_TOP_N_FOR_INCREMENTAL,
    "val_bars": VAL_BARS,
    "test_bars": TEST_BARS,
    "embargo_bars": EMBARGO_BARS,
    "strict_train_only": True,
    "hie_stability_uses_spearman_pruning": False,
    "final_hybrid_uses_spearman_pruning": True,
}

with open(OUTPUT_DIR / "hybrid_hie_corr_config.json", "w", encoding="utf-8") as f:
    json.dump(config_for_audit, f, ensure_ascii=False, indent=2)

print(json.dumps(config_for_audit, ensure_ascii=False, indent=2))


{
  "created_at": "2026-05-14T13:01:42.703098",
  "loader": "KlinesDataLoader.load_data",
  "ohlcv_relative_path": "data\\klines_data\\crypto_240m_bybit_TEST.parquet",
  "main_hie_alpha_out": 0.5,
  "alpha_overlap_reference": 1.0,
  "alpha_overlap_demo_z_windows": [
    72
  ],
  "horizon": 10,
  "trade_cost": 0.0025,
  "z_windows": [
    24,
    48,
    72,
    96
  ],
  "train_blocks": [
    "early_train",
    "mid_train",
    "late_train"
  ],
  "hie_topk_stability_grid": [
    7,
    14,
    20,
    26
  ],
  "main_hie_stability_topk": 14,
  "hie_min_stability_count": 2,
  "n_bins": 15,
  "n_bootstrap": 100,
  "min_bin_size": 40,
  "min_action_count_per_bin": 20,
  "prune_threshold": 0.85,
  "default_final_k": 10,
  "default_corr_core_k": 5,
  "default_hie_target_k": 5,
  "strict_exclude_raw_corr_top_n_for_incremental": null,
  "val_bars": 2000,
  "test_bars": 2000,
  "embargo_bars": 0,
  "strict_train_only": true,
  "hie_stability_uses_spearman_pruning": false,
  "final_hybrid_use

## 2. Project imports

In [3]:
# ---------------------------------------------------------------------
# Standard OHLCV loader.
# ---------------------------------------------------------------------
from data_processing.functions.klines_dataloader import KlinesDataLoader

# ---------------------------------------------------------------------
# Feature engineering imports.
# ---------------------------------------------------------------------
from data_processing.functions.stream_indicators import stream_TA_lib
from data_processing.functions.transform_indicators import transform_indicators_df
from data_processing.functions.rolling_z_score_clip import rolling_z_score_clip_df

# ---------------------------------------------------------------------
# HIE selector imports.
# Different project versions sometimes keep direct counterfactual builder
# in slightly different modules, so we use a safe fallback.
# ---------------------------------------------------------------------
from causal_feature_selector_hie_binary import (
        FeatureSelectionConfig,
        CausalBanditFeatureSelector,
        make_direct_counterfactual_log_binary_success,
    )

print("Imports OK")


Imports OK


## 3. Load OHLCV through `KlinesDataLoader`

In [4]:
def load_ohlcv_with_klines_dataloader() -> pd.DataFrame:
    """Load OHLCV strictly via the project's standard KlinesDataLoader."""
    loader = KlinesDataLoader(symbols=ALL_SYMBOLS)
    df = loader.load_data(
        download_path=OHLCV_RELATIVE_PATH,
        analyse_data=True,
        cleaning=True,
    )

    required_cols = set(META_COLS)
    missing = sorted(required_cols - set(df.columns))
    if missing:
        raise ValueError(f"OHLCV dataframe is missing required meta columns: {missing}")

    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df[df["symbol"].isin(ALL_SYMBOLS)].copy()
    df = df.sort_values(["symbol", "timestamp"]).reset_index(drop=True)

    dup = df.duplicated(["symbol", "timestamp"]).sum()
    if dup:
        raise ValueError(f"Found duplicated (symbol, timestamp) rows after loader: {dup}")

    return df


ohlcv = load_ohlcv_with_klines_dataloader()

print("OHLCV shape:", ohlcv.shape)
print("symbols:", sorted(ohlcv["symbol"].unique()))
print("date range:", ohlcv["timestamp"].min(), "→", ohlcv["timestamp"].max())
display(ohlcv.head())


			Статистика без фильтрации

Количество строк: 51672
Количество столбцов: 8
Количество пропущенных значений: 0

Название столбцов: timestamp, open, high, low, close, volume, turnover, symbol
Название активов: BTCUSDT, DOGEUSDT, ETHUSDT, SOLUSDT, XRPUSDT

Длина временного ряда актива BTCUSDT: 10549
Длина временного ряда актива DOGEUSDT: 10209
Длина временного ряда актива ETHUSDT: 10549
Длина временного ряда актива SOLUSDT: 9903
Длина временного ряда актива XRPUSDT: 10462

Временные рамки ряда каждого актива:
BTCUSDT: 2021-07-05 12:00:00+00:00 - 2026-04-28 12:00:00+00:00
DOGEUSDT: 2021-08-31 04:00:00+00:00 - 2026-04-28 12:00:00+00:00
ETHUSDT: 2021-07-05 12:00:00+00:00 - 2026-04-28 12:00:00+00:00
SOLUSDT: 2021-10-21 04:00:00+00:00 - 2026-04-28 12:00:00+00:00
XRPUSDT: 2021-07-20 00:00:00+00:00 - 2026-04-28 12:00:00+00:00


			Статистика после фильтрации (удаление столбца 'turnover' и выравнивание временных рядов по активам)

Количество строк: 49515
Количество столбцов: 7
Количество пропущ

,timestamp,open,high,low,close,volume,symbol
0,2021-10-21 04:00:00+00:00,65070.85,65247.18,64176.50,65125.25,132.123083,BTCUSDT
1,2021-10-21 08:00:00+00:00,65125.25,66645.39,64259.27,64852.19,175.523964,BTCUSDT
2,2021-10-21 12:00:00+00:00,64852.19,65622.69,62128.53,63029.47,226.384309,BTCUSDT
3,2021-10-21 16:00:00+00:00,63029.47,63659.56,62463.64,62828.36,154.019616,BTCUSDT
4,2021-10-21 20:00:00+00:00,62828.36,63103.99,62077.00,62228.43,150.886863,BTCUSDT


## 4. Helper functions: feature engineering, splits, HIE, ranking, pruning

In [5]:
def alpha_to_tag(alpha: float) -> str:
    if float(alpha) == 0.0:
        return "a0"
    if float(alpha) == 0.5:
        return "a0p5"
    if float(alpha) == 1.0:
        return "a1"
    return f"a{str(alpha).replace('.', 'p')}"


def unique_keep_order(items):
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def jaccard(a, b):
    sa = set(a)
    sb = set(b)
    if not sa and not sb:
        return np.nan
    return len(sa & sb) / len(sa | sb)


def overlap_stats(left, right):
    sl = set(left)
    sr = set(right)
    return {
        "overlap_count": len(sl & sr),
        "jaccard": jaccard(left, right),
        "shared_features": sorted(sl & sr),
        "only_left": sorted(sl - sr),
        "only_right": sorted(sr - sl),
    }


def compute_features_for_z_window(ohlcv_df: pd.DataFrame, z_window: int) -> tuple[pd.DataFrame, list[str]]:
    """
    Compute transformed + rolling-z-scored feature table for one z_window.

    Feature engineering runs on full chronological data, but rolling_z_score uses
    shift_by_one=True, so feature value at t uses only information before t.
    HIE/corr later uses strictly train rows.
    """
    parts = []

    for sym, g in ohlcv_df.groupby("symbol", sort=True):
        g = g.sort_values("timestamp").reset_index(drop=True).copy()

        ind = stream_TA_lib(
            g,
            meta_cols=META_COLS,
            **CONFIG_FOR_INDICATORS,
        )

        transformed = transform_indicators_df(ind, meta_cols=META_COLS)

        zdf = rolling_z_score_clip_df(
            transformed,
            meta_cols=META_COLS,
            window=z_window,
            clip_value=5.0,
            shift_by_one=True,
        )

        parts.append(zdf)

    out = pd.concat(parts, ignore_index=True)
    out = out.sort_values(["symbol", "timestamp"]).reset_index(drop=True)

    feature_cols = [c for c in out.columns if c not in META_COLS]
    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=feature_cols + META_COLS).reset_index(drop=True)

    feature_cols = [c for c in out.columns if c not in META_COLS]
    return out, feature_cols


def chronological_split_by_symbol(df: pd.DataFrame) -> pd.DataFrame:
    """Add phase=train/val/test per symbol using last VAL_BARS and TEST_BARS."""
    parts = []
    for sym, g in df.groupby("symbol", sort=True):
        g = g.sort_values("timestamp").reset_index(drop=True).copy()
        n = len(g)

        if n <= VAL_BARS + TEST_BARS + HORIZON + EMBARGO_BARS:
            raise ValueError(f"Too few rows for symbol={sym}: n={n}")

        test_start = n - TEST_BARS
        val_start = n - TEST_BARS - VAL_BARS
        train_end = val_start - EMBARGO_BARS

        if train_end <= HORIZON:
            raise ValueError(f"Train too short for symbol={sym}: train_end={train_end}")

        phase = np.array(["unused_gap"] * n, dtype=object)
        phase[:train_end] = "train"
        if EMBARGO_BARS > 0:
            phase[train_end:val_start] = "embargo"
        phase[val_start:test_start] = "val"
        phase[test_start:] = "test"

        g["phase"] = phase
        g["row_in_symbol_after_features"] = np.arange(n)
        g["z_split_train_end_index_exclusive"] = train_end
        g["z_split_val_start_index"] = val_start
        g["z_split_test_start_index"] = test_start

        parts.append(g)

    return pd.concat(parts, ignore_index=True)


def assert_train_only(df_part: pd.DataFrame, label: str) -> None:
    """Hard guard: HIE/corr selection must receive only train rows."""
    if "phase" not in df_part.columns:
        raise ValueError(f"{label}: phase column is missing")

    phases = sorted(df_part["phase"].dropna().unique().tolist())
    if phases != ["train"]:
        raise ValueError(f"{label}: input is not train-only. Found phases={phases}")

    if df_part.empty:
        raise ValueError(f"{label}: empty train-only dataframe")


def add_train_blocks(train_df: pd.DataFrame) -> pd.DataFrame:
    """Split train rows into early/mid/late chronological blocks per symbol."""
    assert_train_only(train_df, "add_train_blocks input")

    parts = []
    for sym, g in train_df.groupby("symbol", sort=True):
        g = g.sort_values("timestamp").reset_index(drop=True).copy()
        n = len(g)

        idx_parts = np.array_split(np.arange(n), 3)
        block = np.empty(n, dtype=object)
        for block_name, idx in zip(TRAIN_BLOCKS, idx_parts):
            block[idx] = block_name

        g["train_block"] = block
        parts.append(g)

    out = pd.concat(parts, ignore_index=True)
    assert_train_only(out, "add_train_blocks output")
    return out


def fit_hie_for_regime(
    sample_df: pd.DataFrame,
    feature_cols: list[str],
    prev_position: int,
    alpha_out: float,
    z_window: int,
    sample_name: str,
    random_state_offset: int = 0,
) -> pd.DataFrame:
    """Fit binary HIE for one sample and one regime."""
    assert_train_only(sample_df, f"fit_hie_for_regime z={z_window} sample={sample_name}")

    cf = make_direct_counterfactual_log_binary_success(
        sample_df,
        feature_cols=feature_cols,
        horizon=HORIZON,
        trade_cost=TRADE_COST,
        alpha_out=alpha_out,
        prev_position=prev_position,
        action_col_name="raw_action",
        random_state=SEED + random_state_offset,
    )

    cfg = FeatureSelectionConfig(
        n_bins=N_BINS,
        n_bootstrap=N_BOOTSTRAP,
        min_bin_size=MIN_BIN_SIZE,
        min_action_count_per_bin=MIN_ACTION_COUNT_PER_BIN,
        random_state=SEED + random_state_offset,
    )
    selector = CausalBanditFeatureSelector(cfg)
    scores = selector.fit(
        cf,
        feature_cols=feature_cols,
        action_col="raw_action",
        reward_col="reward",
    )

    regime = "entry" if prev_position == 0 else "exit"
    scores = scores.copy()
    scores["z_window"] = z_window
    scores["sample"] = sample_name
    scores["regime"] = regime
    scores["alpha_out"] = float(alpha_out)
    scores["prev_position"] = int(prev_position)
    return scores


def build_hie_union_unpruned(entry_scores: pd.DataFrame, exit_scores: pd.DataFrame) -> pd.DataFrame:
    """
    Combine entry/exit HIE rankings without Spearman pruning.

    Final union ranking inside one sample/z-window:
        1. best_rank = min(entry_rank, exit_rank), ascending
        2. regime_coverage_topk, descending
           = number of regimes where feature is inside MAIN_HIE_STABILITY_TOPK
        3. best_hie_norm, descending
        4. best_p_value, ascending
        5. mean_rank, ascending
        6. feature name, ascending

    Rationale:
        - min(rank) keeps features that are strong in either entry or exit;
        - coverage rewards features that are useful in both regimes;
        - HIE magnitude and p-value are tie-breakers, not primary criteria.
    """
    e = entry_scores[["feature", "hie_rank", "hie_norm", "hie_p_value", "hie_observed", "hie_null_mean"]].copy()
    x = exit_scores[["feature", "hie_rank", "hie_norm", "hie_p_value", "hie_observed", "hie_null_mean"]].copy()

    e = e.rename(columns={
        "hie_rank": "entry_rank",
        "hie_norm": "entry_hie_norm",
        "hie_p_value": "entry_p_value",
        "hie_observed": "entry_hie_observed",
        "hie_null_mean": "entry_hie_null_mean",
    })
    x = x.rename(columns={
        "hie_rank": "exit_rank",
        "hie_norm": "exit_hie_norm",
        "hie_p_value": "exit_p_value",
        "hie_observed": "exit_hie_observed",
        "hie_null_mean": "exit_hie_null_mean",
    })

    union = e.merge(x, on="feature", how="outer")

    for col in ["entry_rank", "exit_rank"]:
        union[col] = union[col].astype(float)

    union["entry_rank_filled"] = union["entry_rank"].fillna(np.inf)
    union["exit_rank_filled"] = union["exit_rank"].fillna(np.inf)
    union["best_rank"] = union[["entry_rank_filled", "exit_rank_filled"]].min(axis=1)
    union["mean_rank"] = union[["entry_rank_filled", "exit_rank_filled"]].replace(np.inf, np.nan).mean(axis=1)

    union["entry_hie_norm_filled"] = union["entry_hie_norm"].fillna(-np.inf)
    union["exit_hie_norm_filled"] = union["exit_hie_norm"].fillna(-np.inf)
    union["best_hie_norm"] = union[["entry_hie_norm_filled", "exit_hie_norm_filled"]].max(axis=1)

    union["entry_p_value_filled"] = union["entry_p_value"].fillna(1.0)
    union["exit_p_value_filled"] = union["exit_p_value"].fillna(1.0)
    union["best_p_value"] = union[["entry_p_value_filled", "exit_p_value_filled"]].min(axis=1)

    # Coverage is defined relative to the stability candidate pool.
    # It is 2 if the feature is important in both entry and exit, 1 if only in one.
    # Coverage tie-break is computed at the main stability top-K, not a varying sensitivity K.
    union["regime_coverage_topk"] = (
        (union["entry_rank_filled"] <= MAIN_HIE_STABILITY_TOPK).astype(int)
        + (union["exit_rank_filled"] <= MAIN_HIE_STABILITY_TOPK).astype(int)
    )

    def source(row):
        if row["entry_rank_filled"] < row["exit_rank_filled"]:
            return "entry"
        if row["exit_rank_filled"] < row["entry_rank_filled"]:
            return "exit"
        return "both_tie"

    union["best_regime_source"] = union.apply(source, axis=1)

    union = union.sort_values(
        ["best_rank", "regime_coverage_topk", "best_hie_norm", "best_p_value", "mean_rank", "feature"],
        ascending=[True, False, False, True, True, True],
    ).reset_index(drop=True)

    union["union_rank"] = np.arange(1, len(union) + 1)

    drop_cols = [
        "entry_rank_filled", "exit_rank_filled",
        "entry_hie_norm_filled", "exit_hie_norm_filled",
        "entry_p_value_filled", "exit_p_value_filled",
    ]
    union = union.drop(columns=[c for c in drop_cols if c in union.columns])
    return union


def make_train_corr_target(train_df: pd.DataFrame) -> pd.DataFrame:
    """Add future_log_return target for train-only Spearman correlation."""
    assert_train_only(train_df, "make_train_corr_target input")
    parts = []
    for sym, g in train_df.groupby("symbol", sort=True):
        g = g.sort_values("timestamp").reset_index(drop=True).copy()
        g["future_close"] = g["close"].shift(-HORIZON)
        g["future_log_return"] = np.log(g["future_close"] / g["close"])
        g = g.replace([np.inf, -np.inf], np.nan)
        g = g.dropna(subset=["future_log_return"]).copy()
        parts.append(g)
    out = pd.concat(parts, ignore_index=True)
    assert_train_only(out, "make_train_corr_target output")
    return out


def compute_spearman_corr_ranking(
    train_df_with_target: pd.DataFrame,
    feature_cols: list[str],
    target_col: str = "future_log_return",
) -> pd.DataFrame:
    """Compute train-only Spearman correlation ranking with future_log_return."""
    assert_train_only(train_df_with_target, "compute_spearman_corr_ranking input")

    rows = []
    for f in feature_cols:
        if f not in train_df_with_target.columns:
            continue
        x = train_df_with_target[f]
        y = train_df_with_target[target_col]
        valid = x.notna() & y.notna()
        if valid.sum() < 20 or x[valid].nunique() <= 1:
            corr = np.nan
        else:
            corr = x[valid].corr(y[valid], method="spearman")
        rows.append({
            "feature": f,
            "spearman_corr": corr,
            "abs_spearman_corr": abs(corr) if pd.notna(corr) else np.nan,
            "n_obs": int(valid.sum()),
        })

    out = pd.DataFrame(rows).dropna(subset=["abs_spearman_corr"]).copy()
    out = out.sort_values(
        ["abs_spearman_corr", "feature"],
        ascending=[False, True],
    ).reset_index(drop=True)
    out["corr_rank"] = np.arange(1, len(out) + 1)
    return out


def greedy_spearman_prune_ordered(
    candidates: list[str],
    train_feature_df: pd.DataFrame,
    threshold: float = 0.85,
    top_k: int = 10,
    return_diagnostics: bool = False,
):
    """Greedy Spearman pruning preserving candidate order."""
    candidates = unique_keep_order(candidates)
    available = [f for f in candidates if f in train_feature_df.columns]
    missing = [f for f in candidates if f not in train_feature_df.columns]

    if len(available) == 0:
        raise ValueError("No candidate features found in train_feature_df.")

    # Drop bad columns.
    good_cols = []
    bad_cols = []
    for c in available:
        s = train_feature_df[c]
        if s.notna().sum() < 5 or s.nunique(dropna=True) <= 1:
            bad_cols.append(c)
        else:
            good_cols.append(c)
    available = good_cols

    if len(available) == 0:
        raise ValueError("No non-constant candidate features available after filtering.")

    corr = train_feature_df[available].corr(method="spearman").abs()

    selected = []
    diagnostics = []

    for f in available:
        if len(selected) == 0:
            selected.append(f)
            diagnostics.append({
                "feature": f,
                "decision": "selected",
                "reason": "first_feature",
                "max_abs_corr_with_selected": np.nan,
                "most_correlated_selected_feature": None,
            })
        else:
            corr_to_selected = corr.loc[f, selected]
            max_corr = float(corr_to_selected.max())
            most_corr_feature = str(corr_to_selected.idxmax())

            if pd.isna(max_corr) or max_corr < threshold:
                selected.append(f)
                diagnostics.append({
                    "feature": f,
                    "decision": "selected",
                    "reason": "below_threshold",
                    "max_abs_corr_with_selected": max_corr,
                    "most_correlated_selected_feature": most_corr_feature,
                })
            else:
                diagnostics.append({
                    "feature": f,
                    "decision": "dropped",
                    "reason": "too_correlated",
                    "max_abs_corr_with_selected": max_corr,
                    "most_correlated_selected_feature": most_corr_feature,
                })

        if len(selected) >= top_k:
            break

    for f in missing:
        diagnostics.append({
            "feature": f,
            "decision": "missing",
            "reason": "not_in_train_feature_df",
            "max_abs_corr_with_selected": np.nan,
            "most_correlated_selected_feature": None,
        })

    diag = pd.DataFrame(diagnostics)
    if return_diagnostics:
        return selected, diag
    return selected


## 5. Compute features, HIE stability, and train-only corr rankings

Долгий блок. После него всё сохраняется в CSV, и hybrid sizes можно менять без пересчёта HIE.


In [6]:
FEATURE_TABLES_BY_Z = {}
FEATURE_COLS_BY_Z = {}
TRAIN_BLOCKED_BY_Z = {}

HIE_SCORES_MAIN = []
HIE_UNION_RANKINGS_MAIN = []
HIE_TOPK_LONG_ROWS = []

CORR_RANKINGS_BY_Z = {}
CORR_PRUNE_DIAGS_BY_Z = {}
SPLIT_ROWS = []
TRAIN_ONLY_AUDIT_ROWS = []

for z_window in Z_WINDOWS:
    print("\n" + "=" * 110)
    print(f"z_window={z_window}: feature engineering, HIE alpha={MAIN_HIE_ALPHA_OUT}, corr ranking")
    print("=" * 110)

    zdf, feature_cols = compute_features_for_z_window(ohlcv, z_window)
    zdf = chronological_split_by_symbol(zdf)

    # Split audit.
    for sym, g in zdf.groupby("symbol", sort=True):
        g = g.sort_values("timestamp")
        train_g = g[g["phase"] == "train"]
        val_g = g[g["phase"] == "val"]
        test_g = g[g["phase"] == "test"]
        SPLIT_ROWS.append({
            "z_window": z_window,
            "symbol": sym,
            "n_total_after_features": len(g),
            "n_train": len(train_g),
            "n_val": len(val_g),
            "n_test": len(test_g),
            "train_start": train_g["timestamp"].min(),
            "train_end": train_g["timestamp"].max(),
            "val_start": val_g["timestamp"].min(),
            "val_end": val_g["timestamp"].max(),
            "test_start": test_g["timestamp"].min(),
            "test_end": test_g["timestamp"].max(),
        })

    train_df = zdf[zdf["phase"].eq("train")].copy()
    assert_train_only(train_df, f"z={z_window} train_df")
    train_blocked = add_train_blocks(train_df)

    FEATURE_TABLES_BY_Z[z_window] = train_df
    FEATURE_COLS_BY_Z[z_window] = feature_cols
    TRAIN_BLOCKED_BY_Z[z_window] = train_blocked

    # -----------------------------------------------------------------
    # Corr ranking on train.
    # -----------------------------------------------------------------
    train_df_with_target = make_train_corr_target(train_df)
    corr_ranking = compute_spearman_corr_ranking(train_df_with_target, feature_cols)
    corr_ranking["z_window"] = z_window
    CORR_RANKINGS_BY_Z[z_window] = corr_ranking

    corr_ordered = corr_ranking["feature"].tolist()
    corr_pruned_for_audit, corr_prune_diag = greedy_spearman_prune_ordered(
        candidates=corr_ordered,
        train_feature_df=train_df,
        threshold=PRUNE_THRESHOLD,
        top_k=max(30, DEFAULT_FINAL_K),
        return_diagnostics=True,
    )
    corr_prune_diag["z_window"] = z_window
    corr_prune_diag["set_type"] = "corr_pruning_audit_top30"
    CORR_PRUNE_DIAGS_BY_Z[z_window] = corr_prune_diag

    print("feature cols:", len(feature_cols))
    print("train rows:", len(train_df))
    print("corr raw top 10:", corr_ordered[:10])
    print("corr pruned top 10:", corr_pruned_for_audit[:10])

    # -----------------------------------------------------------------
    # HIE for samples: full_train + 3 chronological train blocks.
    # -----------------------------------------------------------------
    sample_specs = [("full_train", train_blocked)]
    for block_name in TRAIN_BLOCKS:
        sample_specs.append((block_name, train_blocked[train_blocked["train_block"].eq(block_name)].copy()))

    for sample_name, sample_df in sample_specs:
        assert_train_only(sample_df, f"z={z_window} sample={sample_name}")

        TRAIN_ONLY_AUDIT_ROWS.append({
            "z_window": z_window,
            "sample": sample_name,
            "n_rows": len(sample_df),
            "n_symbols": sample_df["symbol"].nunique(),
            "min_timestamp": sample_df["timestamp"].min(),
            "max_timestamp": sample_df["timestamp"].max(),
            "phases": ",".join(sorted(sample_df["phase"].unique())),
            "contains_val_or_test": bool(sample_df["phase"].isin(["val", "test"]).any()),
        })

        print(f"  HIE sample={sample_name}: rows={len(sample_df)}")

        entry_scores = fit_hie_for_regime(
            sample_df=sample_df,
            feature_cols=feature_cols,
            prev_position=0,
            alpha_out=MAIN_HIE_ALPHA_OUT,
            z_window=z_window,
            sample_name=sample_name,
            random_state_offset=1000 + z_window,
        )
        exit_scores = fit_hie_for_regime(
            sample_df=sample_df,
            feature_cols=feature_cols,
            prev_position=1,
            alpha_out=MAIN_HIE_ALPHA_OUT,
            z_window=z_window,
            sample_name=sample_name,
            random_state_offset=2000 + z_window,
        )

        HIE_SCORES_MAIN.append(entry_scores)
        HIE_SCORES_MAIN.append(exit_scores)

        union = build_hie_union_unpruned(entry_scores, exit_scores)
        union["z_window"] = z_window
        union["sample"] = sample_name
        union["alpha_out"] = float(MAIN_HIE_ALPHA_OUT)
        HIE_UNION_RANKINGS_MAIN.append(union)

        for k in HIE_TOPK_STABILITY_GRID:
            topk = union.head(k).copy()
            for _, row in topk.iterrows():
                HIE_TOPK_LONG_ROWS.append({
                    "z_window": z_window,
                    "sample": sample_name,
                    "alpha_out": float(MAIN_HIE_ALPHA_OUT),
                    "top_k": k,
                    "feature": row["feature"],
                    "union_rank": int(row["union_rank"]),
                    "best_rank": float(row["best_rank"]),
                    "mean_rank": float(row["mean_rank"]),
                    "best_hie_norm": float(row["best_hie_norm"]),
                    "best_p_value": float(row["best_p_value"]),
                    "regime_coverage_topk": int(row.get("regime_coverage_topk", 0)),
                    "best_regime_source": row["best_regime_source"],
                    "entry_rank": row.get("entry_rank", np.nan),
                    "exit_rank": row.get("exit_rank", np.nan),
                    "entry_hie_norm": row.get("entry_hie_norm", np.nan),
                    "exit_hie_norm": row.get("exit_hie_norm", np.nan),
                })

# Save outputs.
hie_scores_main = pd.concat(HIE_SCORES_MAIN, ignore_index=True)
hie_union_rankings_main = pd.concat(HIE_UNION_RANKINGS_MAIN, ignore_index=True)
hie_topk_long = pd.DataFrame(HIE_TOPK_LONG_ROWS)
corr_rankings_all = pd.concat(CORR_RANKINGS_BY_Z.values(), ignore_index=True)
corr_prune_diagnostics_all = pd.concat(CORR_PRUNE_DIAGS_BY_Z.values(), ignore_index=True)
split_summary = pd.DataFrame(SPLIT_ROWS)
train_only_audit = pd.DataFrame(TRAIN_ONLY_AUDIT_ROWS)

hie_scores_main.to_csv(OUTPUT_DIR / "hie_scores_all_alpha0p5.csv", index=False)
hie_union_rankings_main.to_csv(OUTPUT_DIR / "hie_union_unpruned_rankings_alpha0p5.csv", index=False)
hie_topk_long.to_csv(OUTPUT_DIR / "hie_union_unpruned_topk_long_alpha0p5.csv", index=False)
corr_rankings_all.to_csv(OUTPUT_DIR / "spearman_corr_rankings_train_only.csv", index=False)
corr_prune_diagnostics_all.to_csv(OUTPUT_DIR / "corr_pruning_audit_top30.csv", index=False)
split_summary.to_csv(OUTPUT_DIR / "split_summary_used.csv", index=False)
train_only_audit.to_csv(OUTPUT_DIR / "train_only_audit.csv", index=False)

if train_only_audit["contains_val_or_test"].any():
    raise RuntimeError("Train-only audit failed: val/test rows detected.")

print("Saved HIE/corr base outputs to:", OUTPUT_DIR)
display(corr_rankings_all.groupby("z_window").head(10))



z_window=24: feature engineering, HIE alpha=0.5, corr ranking
feature cols: 54
train rows: 28370
corr raw top 10: ['ret_accel_6_24_z', 'volatility_72_sqrt_z', 'vol_ratio_168_log1p_z', 'vol_ratio_72_log1p_z', 'range_sqrt_z', 'MOM_pct_72_z', 'log_ret_72_z', 'vol_expansion_168_log1p_z', 'ret_accel_24_72_z', 'MOM_pct_30_z']
corr pruned top 10: ['ret_accel_6_24_z', 'volatility_72_sqrt_z', 'vol_ratio_168_log1p_z', 'range_sqrt_z', 'MOM_pct_72_z', 'ret_accel_24_72_z', 'MOM_pct_30_z', 'ADX_14_scaled', 'NATR_14_z', 'volatility_168_sqrt_z']
  HIE sample=full_train: rows=28370
  HIE sample=early_train: rows=9460
  HIE sample=mid_train: rows=9455
  HIE sample=late_train: rows=9455

z_window=48: feature engineering, HIE alpha=0.5, corr ranking
feature cols: 54
train rows: 28250
corr raw top 10: ['range_sqrt_z', 'vol_ratio_72_log1p_z', 'MACD_hist_pct_z', 'vol_expansion_72_log1p_z', 'dist_to_low_24_sqrt_z', 'vol_expansion_168_log1p_z', 'vol_ratio_168_log1p_z', 'MOM_pct_14_z', 'vol_ratio_24_log1p_z', 

,feature,spearman_corr,abs_spearman_corr,n_obs,corr_rank,z_window
0,ret_accel_6_24_z,-0.042399,0.042399,28320,1,24
1,volatility_72_sqrt_z,0.039009,0.039009,28320,2,24
2,vol_ratio_168_log1p_z,0.036467,0.036467,28320,3,24
3,vol_ratio_72_log1p_z,0.036392,0.036392,28320,4,24
4,range_sqrt_z,0.036120,0.036120,28320,5,24
5,MOM_pct_72_z,-0.034323,0.034323,28320,6,24
6,log_ret_72_z,-0.034248,0.034248,28320,7,24
7,vol_expansion_168_log1p_z,0.033486,0.033486,28320,8,24
8,ret_accel_24_72_z,0.033119,0.033119,28320,9,24
9,MOM_pct_30_z,0.032195,0.032195,28320,10,24


In [8]:
# # ============================================================
# # Rebuild hie_topk_long from cached hie_union_rankings_main
# # Fix: include regime_coverage_topk
# # ============================================================
#
# HIE_TOPK_LONG_ROWS_FIXED = []
#
# for _, group in hie_union_rankings_main.groupby(
#     ["z_window", "sample", "alpha_out"],
#     sort=True,
# ):
#     group = group.sort_values("union_rank").copy()
#
#     z_window = int(group["z_window"].iloc[0])
#     sample_name = group["sample"].iloc[0]
#     alpha_out = float(group["alpha_out"].iloc[0])
#
#     for k in HIE_TOPK_STABILITY_GRID:
#         topk = group.head(k).copy()
#
#         for _, row in topk.iterrows():
#             HIE_TOPK_LONG_ROWS_FIXED.append({
#                 "z_window": z_window,
#                 "sample": sample_name,
#                 "alpha_out": alpha_out,
#                 "top_k": k,
#                 "feature": row["feature"],
#                 "union_rank": int(row["union_rank"]),
#                 "best_rank": float(row["best_rank"]),
#                 "mean_rank": float(row["mean_rank"]),
#                 "best_hie_norm": float(row["best_hie_norm"]),
#                 "best_p_value": float(row["best_p_value"]),
#                 "regime_coverage_topk": int(row.get("regime_coverage_topk", 0)),
#                 "best_regime_source": row["best_regime_source"],
#                 "entry_rank": row.get("entry_rank", np.nan),
#                 "exit_rank": row.get("exit_rank", np.nan),
#                 "entry_hie_norm": row.get("entry_hie_norm", np.nan),
#                 "exit_hie_norm": row.get("exit_hie_norm", np.nan),
#             })
#
# hie_topk_long = pd.DataFrame(HIE_TOPK_LONG_ROWS_FIXED)
#
# # Save fixed long table.
# hie_topk_long.to_csv(
#     OUTPUT_DIR / "hie_union_unpruned_topk_long_alpha0p5.csv",
#     index=False,
# )
#
# print("Fixed hie_topk_long shape:", hie_topk_long.shape)
# print("Columns:", hie_topk_long.columns.tolist())
#
# required_cols = [
#     "z_window",
#     "sample",
#     "top_k",
#     "feature",
#     "union_rank",
#     "best_rank",
#     "mean_rank",
#     "best_hie_norm",
#     "best_p_value",
#     "regime_coverage_topk",
#     "best_regime_source",
# ]
#
# missing = [c for c in required_cols if c not in hie_topk_long.columns]
# if missing:
#     raise RuntimeError(f"Still missing columns: {missing}")
#
# display(hie_topk_long.head())

Fixed hie_topk_long shape: (1072, 16)
Columns: ['z_window', 'sample', 'alpha_out', 'top_k', 'feature', 'union_rank', 'best_rank', 'mean_rank', 'best_hie_norm', 'best_p_value', 'regime_coverage_topk', 'best_regime_source', 'entry_rank', 'exit_rank', 'entry_hie_norm', 'exit_hie_norm']


,z_window,sample,alpha_out,top_k,feature,union_rank,best_rank,mean_rank,best_hie_norm,best_p_value,regime_coverage_topk,best_regime_source,entry_rank,exit_rank,entry_hie_norm,exit_hie_norm
0,24,early_train,0.5,7,ADX_72_scaled,1,1.0,1.0,0.026134,0.0,2,both_tie,1.0,1.0,0.006811,0.026134
1,24,early_train,0.5,7,ema_spread_100_200_z,2,2.0,3.5,0.025790,0.0,2,exit,5.0,2.0,0.003599,0.025790
2,24,early_train,0.5,7,dist_to_high_24_sqrt_z,3,2.0,8.5,0.012646,0.0,1,entry,2.0,15.0,0.004775,0.012646
3,24,early_train,0.5,7,ema_spread_50_100_z,4,3.0,5.0,0.022805,0.0,2,exit,7.0,3.0,0.003279,0.022805
4,24,early_train,0.5,7,ADX_30_scaled,5,3.0,6.0,0.015485,0.0,2,entry,3.0,9.0,0.003826,0.015485


## 6. Stability aggregation: per z-window and global

In [9]:
def build_per_z_stability(hie_topk_long: pd.DataFrame, top_k: int = MAIN_HIE_STABILITY_TOPK) -> pd.DataFrame:
    """Per-z stability: how often each feature appears in early/mid/late raw HIE union top-K."""
    d = hie_topk_long[
        (hie_topk_long["top_k"].eq(top_k))
        & (hie_topk_long["sample"].isin(TRAIN_BLOCKS))
    ].copy()

    if d.empty:
        raise ValueError(f"No HIE top-k data for top_k={top_k}")

    agg = (
        d.groupby(["z_window", "feature"], as_index=False)
        .agg(
            stability_count=("sample", "nunique"),
            median_union_rank=("union_rank", "median"),
            mean_union_rank=("union_rank", "mean"),
            min_union_rank=("union_rank", "min"),
            median_best_rank=("best_rank", "median"),
            median_mean_rank=("mean_rank", "median"),
            median_best_hie_norm=("best_hie_norm", "median"),
            mean_best_hie_norm=("best_hie_norm", "mean"),
            median_best_p_value=("best_p_value", "median"),
            median_regime_coverage_topk=("regime_coverage_topk", "median"),
            n_entry_source=("best_regime_source", lambda s: int((s == "entry").sum())),
            n_exit_source=("best_regime_source", lambda s: int((s == "exit").sum())),
            n_both_tie_source=("best_regime_source", lambda s: int((s == "both_tie").sum())),
            samples_seen=("sample", lambda s: "|".join(sorted(s.unique()))),
        )
    )

    full = hie_topk_long[
        (hie_topk_long["top_k"].eq(top_k))
        & (hie_topk_long["sample"].eq("full_train"))
    ][["z_window", "feature", "union_rank", "best_rank", "best_hie_norm", "best_p_value", "regime_coverage_topk"]].copy()
    full = full.rename(columns={
        "union_rank": "full_train_union_rank",
        "best_rank": "full_train_best_rank",
        "best_hie_norm": "full_train_best_hie_norm",
        "best_p_value": "full_train_best_p_value",
        "regime_coverage_topk": "full_train_regime_coverage_topk",
    })

    agg = agg.merge(full, on=["z_window", "feature"], how="left")
    agg["full_train_union_rank_filled"] = agg["full_train_union_rank"].fillna(10_000)
    agg["full_train_best_rank_filled"] = agg["full_train_best_rank"].fillna(10_000)
    agg["full_train_best_hie_norm_filled"] = agg["full_train_best_hie_norm"].fillna(-np.inf)
    agg["full_train_regime_coverage_topk_filled"] = agg["full_train_regime_coverage_topk"].fillna(0)

    # Stable HIE candidate ranking across train blocks.
    # Primary = recurrence, then rank quality, then HIE strength, then full-train confirmation.
    agg = agg.sort_values(
        [
            "z_window",
            "stability_count",
            "median_best_rank",
            "median_regime_coverage_topk",
            "median_best_hie_norm",
            "full_train_best_rank_filled",
            "full_train_regime_coverage_topk_filled",
            "full_train_best_hie_norm_filled",
            "median_mean_rank",
            "feature",
        ],
        ascending=[True, False, True, False, False, True, False, False, True, True],
    ).reset_index(drop=True)

    agg["stable_hie_rank_within_z"] = agg.groupby("z_window").cumcount() + 1
    agg["is_stable_ge2"] = agg["stability_count"] >= 2
    agg["stability_rate"] = agg["stability_count"] / len(TRAIN_BLOCKS)
    agg["stability_top_k"] = top_k
    return agg


def build_global_stability(hie_topk_long: pd.DataFrame, top_k: int = MAIN_HIE_STABILITY_TOPK) -> pd.DataFrame:
    """Global stability across all z_windows and all train blocks."""
    d = hie_topk_long[
        (hie_topk_long["top_k"].eq(top_k))
        & (hie_topk_long["sample"].isin(TRAIN_BLOCKS))
    ].copy()

    agg = (
        d.groupby("feature", as_index=False)
        .agg(
            global_appearance_count=("sample", "count"),
            n_z_windows_seen=("z_window", "nunique"),
            z_windows_seen=("z_window", lambda s: "|".join(map(str, sorted(s.unique())))),
            median_union_rank_global=("union_rank", "median"),
            mean_union_rank_global=("union_rank", "mean"),
            median_best_hie_norm_global=("best_hie_norm", "median"),
            mean_best_hie_norm_global=("best_hie_norm", "mean"),
            n_entry_source_global=("best_regime_source", lambda s: int((s == "entry").sum())),
            n_exit_source_global=("best_regime_source", lambda s: int((s == "exit").sum())),
            n_both_tie_source_global=("best_regime_source", lambda s: int((s == "both_tie").sum())),
        )
    )

    # Per-z count >= 2 summary.
    per_z = build_per_z_stability(hie_topk_long, top_k=top_k)
    per_z_stable = (
        per_z[per_z["is_stable_ge2"]]
        .groupby("feature", as_index=False)
        .agg(
            n_z_windows_stable_ge2=("z_window", "nunique"),
            z_windows_stable_ge2=("z_window", lambda s: "|".join(map(str, sorted(s.unique())))),
        )
    )
    agg = agg.merge(per_z_stable, on="feature", how="left")
    agg["n_z_windows_stable_ge2"] = agg["n_z_windows_stable_ge2"].fillna(0).astype(int)
    agg["z_windows_stable_ge2"] = agg["z_windows_stable_ge2"].fillna("")

    agg = agg.sort_values(
        [
            "global_appearance_count",
            "n_z_windows_stable_ge2",
            "n_z_windows_seen",
            "median_union_rank_global",
            "median_best_hie_norm_global",
            "feature",
        ],
        ascending=[False, False, False, True, False, True],
    ).reset_index(drop=True)
    agg["global_stable_hie_rank"] = np.arange(1, len(agg) + 1)
    agg["stability_top_k"] = top_k
    return agg


STABLE_HIE_PER_Z_BY_K = {}
STABLE_HIE_GLOBAL_BY_K = {}
for k in HIE_TOPK_STABILITY_GRID:
    STABLE_HIE_PER_Z_BY_K[k] = build_per_z_stability(hie_topk_long, top_k=k)
    STABLE_HIE_GLOBAL_BY_K[k] = build_global_stability(hie_topk_long, top_k=k)
    STABLE_HIE_PER_Z_BY_K[k].to_csv(OUTPUT_DIR / f"stable_hie_per_z_top{k}_alpha0p5.csv", index=False)
    STABLE_HIE_GLOBAL_BY_K[k].to_csv(OUTPUT_DIR / f"stable_hie_global_top{k}_alpha0p5.csv", index=False)

stable_hie_per_z_main = STABLE_HIE_PER_Z_BY_K[MAIN_HIE_STABILITY_TOPK]
stable_hie_global_main = STABLE_HIE_GLOBAL_BY_K[MAIN_HIE_STABILITY_TOPK]

stable_hie_per_z_main.to_csv(OUTPUT_DIR / "stable_hie_per_z_MAIN_alpha0p5.csv", index=False)
stable_hie_global_main.to_csv(OUTPUT_DIR / "stable_hie_global_MAIN_alpha0p5.csv", index=False)

print("Per-z stable HIE main top-K:", MAIN_HIE_STABILITY_TOPK)
display(stable_hie_per_z_main.groupby("z_window").head(10))
print("Global stable HIE:")
display(stable_hie_global_main.head(20))


Per-z stable HIE main top-K: 14


,z_window,feature,stability_count,median_union_rank,mean_union_rank,min_union_rank,median_best_rank,median_mean_rank,median_best_hie_norm,mean_best_hie_norm,...,full_train_best_p_value,full_train_regime_coverage_topk,full_train_union_rank_filled,full_train_best_rank_filled,full_train_best_hie_norm_filled,full_train_regime_coverage_topk_filled,stable_hie_rank_within_z,is_stable_ge2,stability_rate,stability_top_k
0,24,ADX_30_scaled,3,5.0,7.000000,4,3.0,6.00,0.017177,0.020136,...,0.0,1.0,4.0,2.0,0.010843,1.0,1,True,1.000000,14
1,24,dist_to_high_24_sqrt_z,3,7.0,6.000000,3,4.0,8.50,0.015101,0.018883,...,NaN,NaN,10000.0,10000.0,-inf,0.0,2,True,1.000000,14
2,24,price_pos_in_range_72_norm,3,9.0,9.666667,6,7.0,12.50,0.018220,0.019961,...,0.0,1.0,2.0,1.0,0.009919,1.0,3,True,1.000000,14
3,24,ema_spread_100_200_z,2,5.5,5.500000,2,4.0,14.00,0.019205,0.019205,...,0.0,1.0,7.0,5.0,0.006919,1.0,4,True,0.666667,14
4,24,RSI_72_bounded,2,6.5,6.500000,3,4.0,7.50,0.015119,0.015119,...,0.0,2.0,1.0,1.0,0.011086,2.0,5,True,0.666667,14
5,24,ema_spread_9_21_z,2,8.0,8.000000,5,5.5,10.00,0.020804,0.020804,...,NaN,NaN,10000.0,10000.0,-inf,0.0,6,True,0.666667,14
6,24,RSI_30_bounded,2,8.5,8.500000,5,5.5,13.50,0.016213,0.016213,...,0.0,2.0,5.0,3.0,0.007611,2.0,7,True,0.666667,14
7,24,ema_spread_50_100_z,2,8.5,8.500000,4,6.0,9.50,0.022543,0.022543,...,0.0,1.0,13.0,9.0,0.005666,1.0,8,True,0.666667,14
8,24,dist_ema_21_z,2,14.0,14.000000,14,11.0,17.00,0.019903,0.019903,...,NaN,NaN,10000.0,10000.0,-inf,0.0,9,True,0.666667,14
9,24,log_ret_72_z,1,1.0,1.000000,1,1.0,1.00,0.045553,0.045553,...,0.0,2.0,10.0,7.0,0.006567,2.0,10,False,0.333333,14


Global stable HIE:


,feature,global_appearance_count,n_z_windows_seen,z_windows_seen,median_union_rank_global,mean_union_rank_global,median_best_hie_norm_global,mean_best_hie_norm_global,n_entry_source_global,n_exit_source_global,n_both_tie_source_global,n_z_windows_stable_ge2,z_windows_stable_ge2,global_stable_hie_rank,stability_top_k
0,ema_spread_100_200_z,9,4,24|48|72|96,7.0,6.666667,0.020419,0.022002,4,5,0,4,24|48|72|96,1,14
1,price_pos_in_range_72_norm,9,4,24|48|72|96,11.0,10.222222,0.018220,0.020987,0,9,0,4,24|48|72|96,2,14
2,ADX_30_scaled,9,4,24|48|72|96,6.0,7.444444,0.017693,0.021726,9,0,0,3,24|48|72,3,14
3,volatility_168_sqrt_z,8,4,24|48|72|96,8.5,8.500000,0.021370,0.022069,5,3,0,3,48|72|96,4,14
4,ret_accel_72_168_z,7,4,24|48|72|96,3.0,3.142857,0.036369,0.034203,0,3,4,3,48|72|96,5,14
5,dist_to_high_168_sqrt_z,7,4,24|48|72|96,9.0,8.285714,0.017574,0.019004,1,6,0,3,48|72|96,6,14
6,MOM_pct_72_z,7,4,24|48|72|96,2.0,4.000000,0.030706,0.031546,2,4,1,2,72|96,7,14
7,log_ret_72_z,7,4,24|48|72|96,3.0,4.428571,0.030318,0.031807,1,4,2,2,72|96,8,14
8,RSI_72_bounded,6,4,24|48|72|96,4.5,5.666667,0.017259,0.015388,5,0,1,2,24|48,9,14
9,ema_spread_50_100_z,6,4,24|48|72|96,8.0,8.833333,0.022543,0.023591,2,4,0,2,24|48,10,14


## 6b. Stability sensitivity by top-K

This block makes the top-K choice explicit for defense. Stability is **not** invariant to K: stable sets are nested and generally grow as K increases. The main hybrid uses `MAIN_HIE_STABILITY_TOPK = 14`, while K=7/20/26 are reported as sensitivity checks.

Outputs:
- `stability_robustness_by_top_k.csv`: stable-set size by z-window and K.
- `stable_hie_core_across_top_k.csv`: strict core stable for all K and loose union stable for any K.
- `stable_hie_pairwise_jaccard_by_top_k.csv`: pairwise overlap between stable sets across K.


In [10]:
# ---------------------------------------------------------------------
# Stability sensitivity diagnostics.
# ---------------------------------------------------------------------
# Why this exists:
#   StableSet(K) is monotone in K. A feature stable at K=7 is also stable at K=14/20/26,
#   but a feature may become stable only when K is widened. Therefore we report sensitivity
#   instead of hiding the dependence on top-K.

stability_robustness_rows = []
for k in HIE_TOPK_STABILITY_GRID:
    per_z = STABLE_HIE_PER_Z_BY_K[k]
    for z in Z_WINDOWS:
        z_part = per_z[per_z["z_window"].eq(z)].copy()
        stable_ge2 = z_part[z_part["stability_count"] >= HIE_MIN_STABILITY_COUNT]
        stability_robustness_rows.append({
            "z_window": z,
            "top_k_stability": k,
            "n_count_3": int((z_part["stability_count"] == 3).sum()),
            "n_count_2": int((z_part["stability_count"] == 2).sum()),
            "n_count_1": int((z_part["stability_count"] == 1).sum()),
            "n_stable_ge_2": int(len(stable_ge2)),
            "stable_set_ge_2": "|".join(stable_ge2["feature"].tolist()),
            "top10_stable_ranking": "|".join(stable_ge2.head(10)["feature"].tolist()),
        })

stability_robustness_df = pd.DataFrame(stability_robustness_rows)
stability_robustness_df.to_csv(
    OUTPUT_DIR / "stability_robustness_by_top_k.csv",
    index=False,
)
print("Stability robustness by K:")
display(stability_robustness_df)

# Strict/loose core across all K values.
stable_core_rows = []
for z in Z_WINDOWS:
    sets_by_k = {}
    for k in HIE_TOPK_STABILITY_GRID:
        per_z = STABLE_HIE_PER_Z_BY_K[k]
        z_part = per_z[
            (per_z["z_window"].eq(z))
            & (per_z["stability_count"] >= HIE_MIN_STABILITY_COUNT)
        ]
        sets_by_k[k] = set(z_part["feature"].tolist())

    stable_all_k = set.intersection(*sets_by_k.values()) if sets_by_k else set()
    stable_any_k = set.union(*sets_by_k.values()) if sets_by_k else set()
    stable_main = sets_by_k.get(MAIN_HIE_STABILITY_TOPK, set())

    stable_core_rows.append({
        "z_window": z,
        "k_values": "|".join(map(str, sorted(sets_by_k.keys()))),
        "main_top_k": MAIN_HIE_STABILITY_TOPK,
        "n_stable_main_k": len(stable_main),
        "stable_main_k": "|".join(sorted(stable_main)),
        "n_stable_all_k": len(stable_all_k),
        "stable_all_k": "|".join(sorted(stable_all_k)),
        "n_stable_any_k": len(stable_any_k),
        "stable_any_k": "|".join(sorted(stable_any_k)),
        "n_only_loose_not_main": len(stable_any_k - stable_main),
        "only_loose_not_main": "|".join(sorted(stable_any_k - stable_main)),
    })

stable_core_df = pd.DataFrame(stable_core_rows)
stable_core_df.to_csv(
    OUTPUT_DIR / "stable_hie_core_across_top_k.csv",
    index=False,
)
print("Stable HIE core across K values:")
display(stable_core_df)

# Pairwise Jaccard between stable sets for each z-window and K-pair.
pairwise_rows = []
for z in Z_WINDOWS:
    sets_by_k = {}
    for k in HIE_TOPK_STABILITY_GRID:
        per_z = STABLE_HIE_PER_Z_BY_K[k]
        z_part = per_z[
            (per_z["z_window"].eq(z))
            & (per_z["stability_count"] >= HIE_MIN_STABILITY_COUNT)
        ]
        sets_by_k[k] = set(z_part["feature"].tolist())

    ks = sorted(sets_by_k)
    for i, k_left in enumerate(ks):
        for k_right in ks[i + 1:]:
            left = sets_by_k[k_left]
            right = sets_by_k[k_right]
            pairwise_rows.append({
                "z_window": z,
                "top_k_left": k_left,
                "top_k_right": k_right,
                "n_left": len(left),
                "n_right": len(right),
                "overlap_count": len(left & right),
                "jaccard": len(left & right) / len(left | right) if (left | right) else np.nan,
                "left_subset_of_right": left.issubset(right),
                "shared_features": "|".join(sorted(left & right)),
                "only_left": "|".join(sorted(left - right)),
                "only_right": "|".join(sorted(right - left)),
            })

stable_pairwise_jaccard_df = pd.DataFrame(pairwise_rows)
stable_pairwise_jaccard_df.to_csv(
    OUTPUT_DIR / "stable_hie_pairwise_jaccard_by_top_k.csv",
    index=False,
)
print("Pairwise stable-set Jaccard across K values:")
display(stable_pairwise_jaccard_df)


Stability robustness by K:


,z_window,top_k_stability,n_count_3,n_count_2,n_count_1,n_stable_ge_2,stable_set_ge_2,top10_stable_ranking
0,24,7,0,2,17,2,ADX_30_scaled|dist_to_high_24_sqrt_z,ADX_30_scaled|dist_to_high_24_sqrt_z
1,48,7,0,3,15,3,RSI_72_bounded|ret_accel_72_168_z|ema_spread_5...,RSI_72_bounded|ret_accel_72_168_z|ema_spread_5...
2,72,7,0,3,15,3,ret_accel_72_168_z|volatility_168_sqrt_z|ema_s...,ret_accel_72_168_z|volatility_168_sqrt_z|ema_s...
3,96,7,2,2,11,4,MOM_pct_72_z|log_ret_72_z|ret_accel_72_168_z|e...,MOM_pct_72_z|log_ret_72_z|ret_accel_72_168_z|e...
4,24,14,3,6,21,9,ADX_30_scaled|dist_to_high_24_sqrt_z|price_pos...,ADX_30_scaled|dist_to_high_24_sqrt_z|price_pos...
5,48,14,2,9,18,11,ADX_30_scaled|bb_width_20_sqrt_z|RSI_72_bounde...,ADX_30_scaled|bb_width_20_sqrt_z|RSI_72_bounde...
6,72,14,1,10,19,11,ema_spread_100_200_z|ret_accel_72_168_z|volati...,ema_spread_100_200_z|ret_accel_72_168_z|volati...
7,96,14,4,6,18,10,MOM_pct_72_z|log_ret_72_z|volatility_168_sqrt_...,MOM_pct_72_z|log_ret_72_z|volatility_168_sqrt_...
8,24,20,6,12,18,18,ADX_30_scaled|dist_to_high_24_sqrt_z|RSI_72_bo...,ADX_30_scaled|dist_to_high_24_sqrt_z|RSI_72_bo...
9,48,20,5,14,17,19,RSI_30_bounded|ADX_30_scaled|price_pos_in_rang...,RSI_30_bounded|ADX_30_scaled|price_pos_in_rang...


Stable HIE core across K values:


,z_window,k_values,main_top_k,n_stable_main_k,stable_main_k,n_stable_all_k,stable_all_k,n_stable_any_k,stable_any_k,n_only_loose_not_main,only_loose_not_main
0,24,7|14|20|26,14,9,ADX_30_scaled|RSI_30_bounded|RSI_72_bounded|di...,2,ADX_30_scaled|dist_to_high_24_sqrt_z,26,ADX_30_scaled|ADX_72_scaled|MOM_pct_72_z|RSI_1...,17,ADX_72_scaled|MOM_pct_72_z|RSI_14_bounded|bb_w...
1,48,7|14|20|26,14,11,ADX_30_scaled|RSI_30_bounded|RSI_72_bounded|bb...,3,RSI_72_bounded|ema_spread_50_100_z|ret_accel_7...,27,ADX_14_scaled|ADX_30_scaled|ADX_72_scaled|MACD...,16,ADX_14_scaled|ADX_72_scaled|MACD_hist_pct_z|NA...
2,72,7|14|20|26,14,11,ADX_30_scaled|MACD_hist_pct_z|MOM_pct_72_z|dis...,3,ema_spread_100_200_z|ret_accel_72_168_z|volati...,28,ADX_30_scaled|ADX_72_scaled|MACD_hist_pct_z|MO...,17,ADX_72_scaled|MOM_pct_30_z|RSI_14_bounded|RSI_...
3,96,7|14|20|26,14,10,MACD_hist_pct_z|MOM_pct_72_z|dist_to_high_168_...,4,MOM_pct_72_z|ema_spread_100_200_z|log_ret_72_z...,26,ADX_30_scaled|ADX_72_scaled|MACD_hist_pct_z|MO...,16,ADX_30_scaled|ADX_72_scaled|MOM_pct_14_z|RSI_1...


Pairwise stable-set Jaccard across K values:


,z_window,top_k_left,top_k_right,n_left,n_right,overlap_count,jaccard,left_subset_of_right,shared_features,only_left,only_right
0,24,7,14,2,9,2,0.222222,True,ADX_30_scaled|dist_to_high_24_sqrt_z,,RSI_30_bounded|RSI_72_bounded|dist_ema_21_z|em...
1,24,7,20,2,18,2,0.111111,True,ADX_30_scaled|dist_to_high_24_sqrt_z,,RSI_30_bounded|RSI_72_bounded|bb_width_20_sqrt...
2,24,7,26,2,26,2,0.076923,True,ADX_30_scaled|dist_to_high_24_sqrt_z,,ADX_72_scaled|MOM_pct_72_z|RSI_14_bounded|RSI_...
3,24,14,20,9,18,9,0.500000,True,ADX_30_scaled|RSI_30_bounded|RSI_72_bounded|di...,,bb_width_20_sqrt_z|dist_ema_100_z|dist_ema_50_...
4,24,14,26,9,26,9,0.346154,True,ADX_30_scaled|RSI_30_bounded|RSI_72_bounded|di...,,ADX_72_scaled|MOM_pct_72_z|RSI_14_bounded|bb_w...
5,24,20,26,18,26,18,0.692308,True,ADX_30_scaled|RSI_30_bounded|RSI_72_bounded|bb...,,ADX_72_scaled|MOM_pct_72_z|RSI_14_bounded|dist...
6,48,7,14,3,11,3,0.272727,True,RSI_72_bounded|ema_spread_50_100_z|ret_accel_7...,,ADX_30_scaled|RSI_30_bounded|bb_width_20_sqrt_...
7,48,7,20,3,19,3,0.157895,True,RSI_72_bounded|ema_spread_50_100_z|ret_accel_7...,,ADX_30_scaled|MACD_hist_pct_z|NATR_14_z|RSI_30...
8,48,7,26,3,27,3,0.111111,True,RSI_72_bounded|ema_spread_50_100_z|ret_accel_7...,,ADX_14_scaled|ADX_30_scaled|ADX_72_scaled|MACD...
9,48,14,20,11,19,11,0.578947,True,ADX_30_scaled|RSI_30_bounded|RSI_72_bounded|bb...,,MACD_hist_pct_z|NATR_14_z|dist_to_high_24_sqrt...


## 7. Alpha overlap demo: HIE full-train alpha=0.5 vs alpha=1.0

Считаем `alpha=1.0` только на `full_train` и только для `ALPHA_OVERLAP_DEMO_Z_WINDOWS`, чтобы показать, что overlap высокий и полный пересчёт по всем блокам не нужен.


In [11]:
ALPHA_OVERLAP_ROWS = []
ALPHA1_UNION_RANKINGS = []

for z_window in ALPHA_OVERLAP_DEMO_Z_WINDOWS:
    print("\n" + "=" * 100)
    print(f"Alpha overlap demo: z_window={z_window}, alpha={MAIN_HIE_ALPHA_OUT} vs {ALPHA_OVERLAP_REFERENCE}")
    print("=" * 100)

    train_df = TRAIN_BLOCKED_BY_Z[z_window]
    feature_cols = FEATURE_COLS_BY_Z[z_window]
    assert_train_only(train_df, f"alpha overlap z={z_window} full_train")

    # Alpha=1.0 full train HIE.
    entry_a1 = fit_hie_for_regime(
        sample_df=train_df,
        feature_cols=feature_cols,
        prev_position=0,
        alpha_out=ALPHA_OVERLAP_REFERENCE,
        z_window=z_window,
        sample_name="full_train_alpha1_demo",
        random_state_offset=1000 + z_window,  # same bootstrap seed as alpha=0.5 entry
    )
    exit_a1 = fit_hie_for_regime(
        sample_df=train_df,
        feature_cols=feature_cols,
        prev_position=1,
        alpha_out=ALPHA_OVERLAP_REFERENCE,
        z_window=z_window,
        sample_name="full_train_alpha1_demo",
        random_state_offset=2000 + z_window,  # same bootstrap seed as alpha=0.5 exit
    )
    union_a1 = build_hie_union_unpruned(entry_a1, exit_a1)
    union_a1["z_window"] = z_window
    union_a1["sample"] = "full_train"
    union_a1["alpha_out"] = float(ALPHA_OVERLAP_REFERENCE)
    ALPHA1_UNION_RANKINGS.append(union_a1)

    union_a05 = hie_union_rankings_main[
        (hie_union_rankings_main["z_window"].eq(z_window))
        & (hie_union_rankings_main["sample"].eq("full_train"))
        & (hie_union_rankings_main["alpha_out"].eq(float(MAIN_HIE_ALPHA_OUT)))
    ].copy()

    # Union top-K overlap.
    for k in HIE_TOPK_STABILITY_GRID:
        a05_feats = union_a05.sort_values("union_rank").head(k)["feature"].tolist()
        a1_feats = union_a1.sort_values("union_rank").head(k)["feature"].tolist()
        sim = overlap_stats(a05_feats, a1_feats)
        ALPHA_OVERLAP_ROWS.append({
            "z_window": z_window,
            "comparison": "union_full_train",
            "top_k": k,
            "alpha_left": MAIN_HIE_ALPHA_OUT,
            "alpha_right": ALPHA_OVERLAP_REFERENCE,
            "overlap_count": sim["overlap_count"],
            "jaccard": sim["jaccard"],
            "shared_features": "|".join(sim["shared_features"]),
            "only_alpha0p5": "|".join(sim["only_left"]),
            "only_alpha1": "|".join(sim["only_right"]),
        })

    # Entry and exit top-13 overlap separately.
    entry_a05 = hie_scores_main[
        (hie_scores_main["z_window"].eq(z_window))
        & (hie_scores_main["sample"].eq("full_train"))
        & (hie_scores_main["regime"].eq("entry"))
    ].sort_values("hie_rank")
    exit_a05 = hie_scores_main[
        (hie_scores_main["z_window"].eq(z_window))
        & (hie_scores_main["sample"].eq("full_train"))
        & (hie_scores_main["regime"].eq("exit"))
    ].sort_values("hie_rank")

    for regime, left_df, right_df in [
        ("entry", entry_a05, entry_a1),
        ("exit", exit_a05, exit_a1),
    ]:
        for k in [7, 14]:
            left_feats = left_df.head(k)["feature"].tolist()
            right_feats = right_df.sort_values("hie_rank").head(k)["feature"].tolist()
            sim = overlap_stats(left_feats, right_feats)
            ALPHA_OVERLAP_ROWS.append({
                "z_window": z_window,
                "comparison": f"{regime}_full_train",
                "top_k": k,
                "alpha_left": MAIN_HIE_ALPHA_OUT,
                "alpha_right": ALPHA_OVERLAP_REFERENCE,
                "overlap_count": sim["overlap_count"],
                "jaccard": sim["jaccard"],
                "shared_features": "|".join(sim["shared_features"]),
                "only_alpha0p5": "|".join(sim["only_left"]),
                "only_alpha1": "|".join(sim["only_right"]),
            })

alpha_overlap_df = pd.DataFrame(ALPHA_OVERLAP_ROWS)
alpha_overlap_df.to_csv(OUTPUT_DIR / "hie_alpha0p5_vs_alpha1_full_train_overlap_demo.csv", index=False)

if ALPHA1_UNION_RANKINGS:
    alpha1_union_rankings = pd.concat(ALPHA1_UNION_RANKINGS, ignore_index=True)
    alpha1_union_rankings.to_csv(OUTPUT_DIR / "hie_union_unpruned_rankings_alpha1_full_train_demo.csv", index=False)

print("Alpha overlap demo saved.")
display(alpha_overlap_df)



Alpha overlap demo: z_window=72, alpha=0.5 vs 1.0
Alpha overlap demo saved.


,z_window,comparison,top_k,alpha_left,alpha_right,overlap_count,jaccard,shared_features,only_alpha0p5,only_alpha1
0,72,union_full_train,7,0.5,1.0,6,0.750000,ADX_30_scaled|RSI_14_bounded|RSI_72_bounded|di...,price_pos_in_range_72_norm,volatility_168_sqrt_z
1,72,union_full_train,14,0.5,1.0,11,0.647059,ADX_30_scaled|ADX_72_scaled|RSI_14_bounded|RSI...,NATR_14_z|dist_ema_200_z|ema_slope_200_z,ema_spread_21_50_z|ema_spread_50_100_z|log_ret...
2,72,union_full_train,20,0.5,1.0,18,0.818182,ADX_30_scaled|ADX_72_scaled|RSI_14_bounded|RSI...,NATR_14_z|ret_accel_72_168_z,ema_spread_21_50_z|price_pos_in_range_168_norm
3,72,union_full_train,26,0.5,1.0,26,1.000000,ADX_14_scaled|ADX_30_scaled|ADX_72_scaled|NATR...,,
4,72,entry_full_train,7,0.5,1.0,7,1.000000,ADX_30_scaled|ADX_72_scaled|RSI_30_bounded|RSI...,,
5,72,entry_full_train,14,0.5,1.0,14,1.000000,ADX_30_scaled|ADX_72_scaled|RSI_14_bounded|RSI...,,
6,72,exit_full_train,7,0.5,1.0,4,0.400000,RSI_14_bounded|RSI_72_bounded|dist_to_high_168...,dist_ema_200_z|ema_slope_200_z|price_pos_in_ra...,ADX_72_scaled|log_ret_6_z|volatility_168_sqrt_z
7,72,exit_full_train,14,0.5,1.0,10,0.555556,ADX_72_scaled|RSI_14_bounded|RSI_30_bounded|RS...,NATR_14_z|ema_slope_200_z|ret_accel_24_72_z|re...,dist_to_low_72_sqrt_z|ema_spread_21_50_z|price...


## 8. Hybrid builder from cached HIE stability and corr ranking

Этот блок можно переиспользовать без пересчёта HIE. Он строит corr baseline и hybrid для любых размеров.


In [12]:
def get_corr_pruned_for_z(z_window: int, final_k: int, prune_threshold: float = PRUNE_THRESHOLD):
    train_df = FEATURE_TABLES_BY_Z[z_window]
    corr_ranking = CORR_RANKINGS_BY_Z[z_window]
    corr_ordered = corr_ranking["feature"].tolist()
    corr_pruned, corr_diag = greedy_spearman_prune_ordered(
        candidates=corr_ordered,
        train_feature_df=train_df,
        threshold=prune_threshold,
        top_k=final_k,
        return_diagnostics=True,
    )
    if len(corr_pruned) < final_k:
        raise RuntimeError(f"z={z_window}: corr_pruned has {len(corr_pruned)} features, need {final_k}")
    corr_diag["z_window"] = z_window
    corr_diag["set_type"] = f"corr_pruned_top{final_k}"
    return corr_pruned, corr_diag


def get_stable_hie_candidates_for_z(
    z_window: int,
    stability_top_k: int = MAIN_HIE_STABILITY_TOPK,
    min_stability_count: int = HIE_MIN_STABILITY_COUNT,
) -> pd.DataFrame:
    per_z = STABLE_HIE_PER_Z_BY_K[stability_top_k].copy()
    part = per_z[
        (per_z["z_window"].eq(z_window))
        & (per_z["stability_count"] >= min_stability_count)
    ].copy()
    return part.sort_values("stable_hie_rank_within_z")


def classify_hie_vs_corr(
    feature: str,
    corr_ranking: pd.DataFrame,
    corr_pruned: list[str],
    corr_prune_diag: pd.DataFrame,
):
    """Audit one stable HIE feature against raw and pruned correlation rankings.

    We intentionally keep only raw corr rank/value, not top10/top15 flags.
    """
    corr_rank_map = corr_ranking.set_index("feature")["corr_rank"].to_dict()
    corr_abs_map = corr_ranking.set_index("feature")["abs_spearman_corr"].to_dict()

    raw_rank = corr_rank_map.get(feature, np.nan)
    abs_corr = corr_abs_map.get(feature, np.nan)

    diag_row = corr_prune_diag[corr_prune_diag["feature"].eq(feature)]
    if not diag_row.empty:
        corr_prune_decision = str(diag_row.iloc[0].get("decision", "unknown"))
        corr_prune_reason = str(diag_row.iloc[0].get("reason", "unknown"))
        corr_prune_max_corr = diag_row.iloc[0].get("max_abs_corr_with_selected", np.nan)
        corr_prune_most_corr = diag_row.iloc[0].get("most_correlated_selected_feature", None)
    else:
        corr_prune_decision = "not_reached_by_corr_pruning_audit"
        corr_prune_reason = "not_reached_by_corr_pruning_audit"
        corr_prune_max_corr = np.nan
        corr_prune_most_corr = None

    return {
        "corr_raw_rank": raw_rank,
        "abs_spearman_corr": abs_corr,
        "in_corr_pruned": feature in corr_pruned,
        "corr_prune_decision_in_audit": corr_prune_decision,
        "corr_prune_reason_in_audit": corr_prune_reason,
        "corr_prune_max_corr_with_selected": corr_prune_max_corr,
        "corr_prune_most_correlated_selected_feature": corr_prune_most_corr,
        "corr_pruning_removed_hie_candidate": (pd.notna(raw_rank) and corr_prune_decision == "dropped"),
    }

def build_hybrid_for_z_from_cached(
    z_window: int,
    final_k: int = DEFAULT_FINAL_K,
    corr_core_k: int = DEFAULT_CORR_CORE_K,
    hie_target_k: int = DEFAULT_HIE_TARGET_K,
    stability_top_k: int = MAIN_HIE_STABILITY_TOPK,
    min_stability_count: int = HIE_MIN_STABILITY_COUNT,
    prune_threshold: float = PRUNE_THRESHOLD,
    strict_exclude_raw_corr_top_n_for_incremental: int | None = STRICT_EXCLUDE_RAW_CORR_TOP_N_FOR_INCREMENTAL,
):
    """
    Build corrPrunedTopK and hybridTopK for one z_window using cached HIE/corr data.

    Hybrid logic:
      1. corr_pruned_topK is baseline.
      2. corr_core = first corr_core_k from corr_pruned_topK.
      3. HIE incremental candidates = stable HIE not in corr_pruned_topK.
         Optional strict filter: also not in raw corr top-N.
      4. Greedy hybrid selection:
         - select corr_core first;
         - try to keep up to hie_target_k HIE incremental features;
         - fill remaining slots from corr fallback;
         - if needed, fill with remaining stable HIE.
    """
    if corr_core_k > final_k:
        raise ValueError("corr_core_k cannot exceed final_k")
    if hie_target_k > final_k:
        raise ValueError("hie_target_k cannot exceed final_k")

    train_df = FEATURE_TABLES_BY_Z[z_window]
    corr_ranking = CORR_RANKINGS_BY_Z[z_window]
    corr_ordered = corr_ranking["feature"].tolist()

    # Corr baseline of the same size as hybrid for fair comparison.
    corr_pruned, corr_diag = get_corr_pruned_for_z(z_window, final_k=final_k, prune_threshold=prune_threshold)
    corr_core = corr_pruned[:corr_core_k]

    stable_df = get_stable_hie_candidates_for_z(
        z_window=z_window,
        stability_top_k=stability_top_k,
        min_stability_count=min_stability_count,
    )
    stable_ordered = stable_df["feature"].tolist()

    corr_rank_map = corr_ranking.set_index("feature")["corr_rank"].to_dict()

    # Main HIE incremental relative to practical pruned corr baseline.
    hie_incremental_pool = []
    hie_excluded_by_raw_corr_strict = []
    for f in stable_ordered:
        if f in corr_pruned:
            continue
        if strict_exclude_raw_corr_top_n_for_incremental is not None:
            rank = corr_rank_map.get(f, np.inf)
            if rank <= strict_exclude_raw_corr_top_n_for_incremental:
                hie_excluded_by_raw_corr_strict.append(f)
                continue
        hie_incremental_pool.append(f)

    # Build final hybrid with HIE target quota.
    selected = []
    selected_sources = {}
    selection_steps = []

    def can_add(feature):
        if feature not in train_df.columns:
            return False, "missing", np.nan, None
        if len(selected) == 0:
            return True, "first_feature", np.nan, None
        cols = selected + [feature]
        corr = train_df[cols].corr(method="spearman").abs()
        corr_to_selected = corr.loc[feature, selected]
        max_corr = float(corr_to_selected.max())
        most_corr = str(corr_to_selected.idxmax())
        if pd.isna(max_corr) or max_corr < prune_threshold:
            return True, "below_threshold", max_corr, most_corr
        return False, "too_correlated", max_corr, most_corr

    def try_add(feature, source, phase):
        nonlocal selected
        if feature in selected:
            selection_steps.append({
                "feature": feature,
                "source": source,
                "phase": phase,
                "decision": "skipped_duplicate",
                "reason": "already_selected",
                "max_abs_corr_with_selected": np.nan,
                "most_correlated_selected_feature": None,
            })
            return False
        ok, reason, max_corr, most_corr = can_add(feature)
        decision = "selected" if ok else "dropped"
        selection_steps.append({
            "feature": feature,
            "source": source,
            "phase": phase,
            "decision": decision,
            "reason": reason,
            "max_abs_corr_with_selected": max_corr,
            "most_correlated_selected_feature": most_corr,
        })
        if ok:
            selected.append(feature)
            selected_sources[feature] = source
            return True
        return False

    # Phase 1: corr core.
    for f in corr_core:
        try_add(f, "corr_core", "corr_core")
        if len(selected) >= final_k:
            break

    # Phase 2: HIE incremental until target kept or pool exhausted.
    hie_kept_count = 0
    for f in hie_incremental_pool:
        if len(selected) >= final_k:
            break
        if hie_kept_count >= hie_target_k:
            break
        if try_add(f, "hie_incremental", "hie_incremental_priority"):
            hie_kept_count += 1

    # Phase 3: fill with corr fallback from raw corr ranking.
    corr_fallback = [f for f in corr_ordered if f not in selected]
    for f in corr_fallback:
        if len(selected) >= final_k:
            break
        try_add(f, "corr_fallback", "corr_fallback_fill")

    # Phase 4: if still not enough, fill with remaining stable HIE.
    remaining_stable_hie = [f for f in stable_ordered if f not in selected]
    for f in remaining_stable_hie:
        if len(selected) >= final_k:
            break
        try_add(f, "hie_fallback", "hie_fallback_fill")

    if len(selected) < final_k:
        raise RuntimeError(f"z={z_window}: hybrid selected only {len(selected)} features; need {final_k}")

    hybrid = selected[:final_k]
    sim = overlap_stats(hybrid, corr_pruned)

    # HIE audit for all stable candidates.
    audit_rows = []
    for _, row in stable_df.iterrows():
        f = row["feature"]
        corr_info = classify_hie_vs_corr(
            feature=f,
            corr_ranking=corr_ranking,
            corr_pruned=corr_pruned,
            corr_prune_diag=CORR_PRUNE_DIAGS_BY_Z[z_window],
        )
        if f in corr_pruned:
            category = "shared_with_corr_pruned"
        elif corr_info.get("corr_prune_decision_in_audit") == "dropped":
            category = "corr_ranked_but_removed_by_pruning"
        elif pd.notna(corr_info.get("corr_raw_rank")):
            category = "not_in_corr_pruned_raw_rank_available"
        else:
            category = "missing_from_corr_ranking"

        selected_in_hybrid = f in hybrid
        source_in_hybrid = selected_sources.get(f, None)
        audit_rows.append({
            "z_window": z_window,
            "feature": f,
            "stable_hie_rank_within_z": row["stable_hie_rank_within_z"],
            "stability_count": row["stability_count"],
            "stability_rate": row["stability_rate"],
            "median_union_rank": row["median_union_rank"],
            "median_best_hie_norm": row["median_best_hie_norm"],
            "full_train_union_rank": row.get("full_train_union_rank", np.nan),
            "selected_in_hybrid": selected_in_hybrid,
            "source_in_hybrid": source_in_hybrid,
            "is_hie_incremental_pool": f in hie_incremental_pool,
            "strict_excluded_by_raw_corr_filter": f in hie_excluded_by_raw_corr_strict,
            "hie_attribution_category": category,
            **corr_info,
        })

    result = {
        "z_window": z_window,
        "final_k": final_k,
        "corr_core_k": corr_core_k,
        "hie_target_k": hie_target_k,
        "stability_top_k": stability_top_k,
        "min_stability_count": min_stability_count,
        "strict_exclude_raw_corr_top_n_for_incremental": strict_exclude_raw_corr_top_n_for_incremental,
        "corr_pruned": corr_pruned,
        "corr_core": corr_core,
        "stable_hie_ordered": stable_ordered,
        "hie_incremental_pool": hie_incremental_pool,
        "hie_excluded_by_raw_corr_strict": hie_excluded_by_raw_corr_strict,
        "hybrid": hybrid,
        "selected_sources": selected_sources,
        "selection_steps": pd.DataFrame(selection_steps),
        "hie_audit": pd.DataFrame(audit_rows),
        "overlap_count": sim["overlap_count"],
        "jaccard": sim["jaccard"],
        "shared_features": sim["shared_features"],
        "only_hybrid": sim["only_left"],
        "only_corr": sim["only_right"],
        "n_hie_incremental_kept": sum(1 for f in hybrid if selected_sources.get(f) == "hie_incremental"),
        "n_corr_core_kept": sum(1 for f in hybrid if selected_sources.get(f) == "corr_core"),
        "n_corr_fallback_kept": sum(1 for f in hybrid if selected_sources.get(f) == "corr_fallback"),
        "n_hie_fallback_kept": sum(1 for f in hybrid if selected_sources.get(f) == "hie_fallback"),
    }
    return result


def build_hybrid_all_z_from_cached(
    scenario_name: str,
    final_k: int = DEFAULT_FINAL_K,
    corr_core_k: int = DEFAULT_CORR_CORE_K,
    hie_target_k: int = DEFAULT_HIE_TARGET_K,
    stability_top_k: int = MAIN_HIE_STABILITY_TOPK,
    min_stability_count: int = HIE_MIN_STABILITY_COUNT,
    prune_threshold: float = PRUNE_THRESHOLD,
    strict_exclude_raw_corr_top_n_for_incremental: int | None = STRICT_EXCLUDE_RAW_CORR_TOP_N_FOR_INCREMENTAL,
):
    results = []
    steps = []
    audits = []

    for z in Z_WINDOWS:
        r = build_hybrid_for_z_from_cached(
            z_window=z,
            final_k=final_k,
            corr_core_k=corr_core_k,
            hie_target_k=hie_target_k,
            stability_top_k=stability_top_k,
            min_stability_count=min_stability_count,
            prune_threshold=prune_threshold,
            strict_exclude_raw_corr_top_n_for_incremental=strict_exclude_raw_corr_top_n_for_incremental,
        )
        results.append(r)

        st = r["selection_steps"].copy()
        st["scenario"] = scenario_name
        st["z_window"] = z
        steps.append(st)

        au = r["hie_audit"].copy()
        au["scenario"] = scenario_name
        audits.append(au)

    summary = pd.DataFrame([
        {
            "scenario": scenario_name,
            "z_window": r["z_window"],
            "final_k": r["final_k"],
            "corr_core_k": r["corr_core_k"],
            "hie_target_k": r["hie_target_k"],
            "stability_top_k": r["stability_top_k"],
            "min_stability_count": r["min_stability_count"],
            "strict_exclude_raw_corr_top_n_for_incremental": r["strict_exclude_raw_corr_top_n_for_incremental"],
            "corr_pruned": "|".join(r["corr_pruned"]),
            "corr_core": "|".join(r["corr_core"]),
            "hie_incremental_pool": "|".join(r["hie_incremental_pool"]),
            "hybrid": "|".join(r["hybrid"]),
            "overlap_count": r["overlap_count"],
            "jaccard": r["jaccard"],
            "shared_features": "|".join(r["shared_features"]),
            "only_hybrid": "|".join(r["only_hybrid"]),
            "only_corr": "|".join(r["only_corr"]),
            "n_hie_incremental_kept": r["n_hie_incremental_kept"],
            "n_corr_core_kept": r["n_corr_core_kept"],
            "n_corr_fallback_kept": r["n_corr_fallback_kept"],
            "n_hie_fallback_kept": r["n_hie_fallback_kept"],
        }
        for r in results
    ])

    steps_df = pd.concat(steps, ignore_index=True) if steps else pd.DataFrame()
    audit_df = pd.concat(audits, ignore_index=True) if audits else pd.DataFrame()

    return results, summary, steps_df, audit_df


## 9. Main scenario: Top10 hybrid 5/5

In [13]:
MAIN_SCENARIO_NAME = "hybrid_top10_corr5_hie5_alpha0p5"

main_results, main_summary, main_steps, main_hie_audit = build_hybrid_all_z_from_cached(
    scenario_name=MAIN_SCENARIO_NAME,
    final_k=10,
    corr_core_k=5,
    hie_target_k=5,
    stability_top_k=MAIN_HIE_STABILITY_TOPK,
    min_stability_count=HIE_MIN_STABILITY_COUNT,
    prune_threshold=PRUNE_THRESHOLD,
    strict_exclude_raw_corr_top_n_for_incremental=STRICT_EXCLUDE_RAW_CORR_TOP_N_FOR_INCREMENTAL,
)

main_summary.to_csv(OUTPUT_DIR / f"{MAIN_SCENARIO_NAME}_summary.csv", index=False)
main_steps.to_csv(OUTPUT_DIR / f"{MAIN_SCENARIO_NAME}_selection_steps.csv", index=False)
main_hie_audit.to_csv(OUTPUT_DIR / f"{MAIN_SCENARIO_NAME}_hie_vs_corr_audit.csv", index=False)

print("Main scenario summary:")
display(main_summary)
print("HIE audit sample:")
display(main_hie_audit.head(30))


Main scenario summary:


,scenario,z_window,final_k,corr_core_k,hie_target_k,stability_top_k,min_stability_count,strict_exclude_raw_corr_top_n_for_incremental,corr_pruned,corr_core,...,hybrid,overlap_count,jaccard,shared_features,only_hybrid,only_corr,n_hie_incremental_kept,n_corr_core_kept,n_corr_fallback_kept,n_hie_fallback_kept
0,hybrid_top10_corr5_hie5_alpha0p5,24,10,5,5,14,2,None,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...,...,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...,5,0.333333,MOM_pct_72_z|range_sqrt_z|ret_accel_6_24_z|vol...,ADX_30_scaled|RSI_72_bounded|dist_to_high_24_s...,ADX_14_scaled|MOM_pct_30_z|NATR_14_z|ret_accel...,5,5,0,0
1,hybrid_top10_corr5_hie5_alpha0p5,48,10,5,5,14,2,None,range_sqrt_z|vol_ratio_72_log1p_z|MACD_hist_pc...,range_sqrt_z|vol_ratio_72_log1p_z|MACD_hist_pc...,...,range_sqrt_z|vol_ratio_72_log1p_z|MACD_hist_pc...,5,0.333333,MACD_hist_pct_z|MOM_pct_14_z|dist_to_low_24_sq...,ADX_30_scaled|RSI_72_bounded|bb_width_20_sqrt_...,ADX_14_scaled|NATR_14_z|body_sqrt_z|ema_spread...,5,5,0,0
2,hybrid_top10_corr5_hie5_alpha0p5,72,10,5,5,14,2,None,range_sqrt_z|vol_ratio_72_log1p_z|dist_to_high...,range_sqrt_z|vol_ratio_72_log1p_z|dist_to_high...,...,range_sqrt_z|vol_ratio_72_log1p_z|dist_to_high...,5,0.333333,bb_width_20_sqrt_z|body_sqrt_z|dist_to_high_16...,ema_slope_100_z|ema_spread_100_200_z|log_ret_7...,ADX_14_scaled|MACD_hist_pct_z|MOM_pct_14_z|NAT...,5,5,0,0
3,hybrid_top10_corr5_hie5_alpha0p5,96,10,5,5,14,2,None,range_sqrt_z|vol_ratio_24_log1p_z|vol_expansio...,range_sqrt_z|vol_ratio_24_log1p_z|vol_expansio...,...,range_sqrt_z|vol_ratio_24_log1p_z|vol_expansio...,5,0.333333,body_sqrt_z|range_sqrt_z|vol_expansion_24_log1...,MOM_pct_72_z|ema_spread_100_200_z|ret_accel_24...,ADX_14_scaled|NATR_14_z|dist_to_high_168_sqrt_...,5,5,0,0


HIE audit sample:


,z_window,feature,stable_hie_rank_within_z,stability_count,stability_rate,median_union_rank,median_best_hie_norm,full_train_union_rank,selected_in_hybrid,source_in_hybrid,...,hie_attribution_category,corr_raw_rank,abs_spearman_corr,in_corr_pruned,corr_prune_decision_in_audit,corr_prune_reason_in_audit,corr_prune_max_corr_with_selected,corr_prune_most_correlated_selected_feature,corr_pruning_removed_hie_candidate,scenario
0,24,ADX_30_scaled,1,3,1.000000,5.0,0.017177,4.0,True,hie_incremental,...,not_in_corr_pruned_raw_rank_available,25,0.022752,False,selected,below_threshold,0.657630,ADX_14_scaled,False,hybrid_top10_corr5_hie5_alpha0p5
1,24,dist_to_high_24_sqrt_z,2,3,1.000000,7.0,0.015101,NaN,True,hie_incremental,...,not_in_corr_pruned_raw_rank_available,29,0.017549,False,selected,below_threshold,0.698115,MOM_pct_14_z,False,hybrid_top10_corr5_hie5_alpha0p5
2,24,price_pos_in_range_72_norm,3,3,1.000000,9.0,0.018220,2.0,True,hie_incremental,...,not_in_corr_pruned_raw_rank_available,37,0.010020,False,selected,below_threshold,0.501143,MOM_pct_72_z,False,hybrid_top10_corr5_hie5_alpha0p5
3,24,ema_spread_100_200_z,4,2,0.666667,5.5,0.019205,7.0,True,hie_incremental,...,not_in_corr_pruned_raw_rank_available,43,0.008623,False,selected,below_threshold,0.695532,price_pos_in_range_72_norm,False,hybrid_top10_corr5_hie5_alpha0p5
4,24,RSI_72_bounded,5,2,0.666667,6.5,0.015119,1.0,True,hie_incremental,...,not_in_corr_pruned_raw_rank_available,48,0.004123,False,not_reached_by_corr_pruning_audit,not_reached_by_corr_pruning_audit,NaN,NaN,False,hybrid_top10_corr5_hie5_alpha0p5
5,24,ema_spread_9_21_z,6,2,0.666667,8.0,0.020804,NaN,False,NaN,...,not_in_corr_pruned_raw_rank_available,17,0.026908,False,selected,below_threshold,0.844024,log_ret_24_z,False,hybrid_top10_corr5_hie5_alpha0p5
6,24,RSI_30_bounded,7,2,0.666667,8.5,0.016213,5.0,False,NaN,...,corr_ranked_but_removed_by_pruning,40,0.009238,False,dropped,too_correlated,0.908008,price_pos_in_range_72_norm,True,hybrid_top10_corr5_hie5_alpha0p5
7,24,ema_spread_50_100_z,8,2,0.666667,8.5,0.022543,13.0,False,NaN,...,not_in_corr_pruned_raw_rank_available,47,0.004299,False,selected,below_threshold,0.756080,ema_spread_21_50_z,False,hybrid_top10_corr5_hie5_alpha0p5
8,24,dist_ema_21_z,9,2,0.666667,14.0,0.019903,NaN,False,NaN,...,corr_ranked_but_removed_by_pruning,41,0.009212,False,dropped,too_correlated,0.854616,MOM_pct_14_z,True,hybrid_top10_corr5_hie5_alpha0p5
9,48,ADX_30_scaled,1,3,1.000000,10.0,0.017693,1.0,True,hie_incremental,...,not_in_corr_pruned_raw_rank_available,27,0.022405,False,selected,below_threshold,0.657622,ADX_14_scaled,False,hybrid_top10_corr5_hie5_alpha0p5


## 10. Save feature sets and metadata for main scenario

Создаём `corr_pruned` и `hybrid` feature sets под `alpha_out=0.5`.


In [14]:
FEATURE_SETS = {}
FEATURE_SET_META = {}

for r in main_results:
    z = int(r["z_window"])
    alpha_tag = alpha_to_tag(MAIN_HIE_ALPHA_OUT)

    corr_name = f"z{z}_{alpha_tag}_corr_pruned_top{r['final_k']}"
    hybrid_name = f"z{z}_{alpha_tag}_hybrid_corr{r['corr_core_k']}_stablehie{r['hie_target_k']}_top{r['final_k']}"

    FEATURE_SETS[corr_name] = r["corr_pruned"]
    FEATURE_SETS[hybrid_name] = r["hybrid"]

    FEATURE_SET_META[corr_name] = {
        "set_name": corr_name,
        "family": "corr_pruned",
        "z_window": z,
        "alpha_out": float(MAIN_HIE_ALPHA_OUT),
        "n_features": len(r["corr_pruned"]),
        "prune_threshold": PRUNE_THRESHOLD,
        "features": r["corr_pruned"],
        "notes": "Train-only Spearman correlation ranking pruned by greedy Spearman threshold.",
    }

    FEATURE_SET_META[hybrid_name] = {
        "set_name": hybrid_name,
        "family": "hybrid_corr_stable_hie",
        "z_window": z,
        "alpha_out": float(MAIN_HIE_ALPHA_OUT),
        "n_features": len(r["hybrid"]),
        "prune_threshold": PRUNE_THRESHOLD,
        "stability_top_k": r["stability_top_k"],
        "min_stability_count": r["min_stability_count"],
        "corr_core_k": r["corr_core_k"],
        "hie_target_k": r["hie_target_k"],
        "strict_exclude_raw_corr_top_n_for_incremental": r["strict_exclude_raw_corr_top_n_for_incremental"],
        "corr_pruned_reference": r["corr_pruned"],
        "corr_core": r["corr_core"],
        "hie_incremental_pool": r["hie_incremental_pool"],
        "n_hie_incremental_kept": r["n_hie_incremental_kept"],
        "n_corr_core_kept": r["n_corr_core_kept"],
        "n_corr_fallback_kept": r["n_corr_fallback_kept"],
        "n_hie_fallback_kept": r["n_hie_fallback_kept"],
        "hybrid_vs_corr_jaccard": r["jaccard"],
        "hybrid_vs_corr_overlap_count": r["overlap_count"],
        "shared_features_with_corr": r["shared_features"],
        "only_hybrid_vs_corr": r["only_hybrid"],
        "only_corr_vs_hybrid": r["only_corr"],
        "features": r["hybrid"],
        "notes": (
            "Hybrid set. HIE stability is measured on unpruned HIE union top-K across early/mid/late train blocks. "
            "Spearman pruning is applied only at the final feature-set construction stage. "
            "HIE audit reports raw corr ranks to distinguish HIE-distinct features from corr-accessible features removed by pruning."
        ),
    }

with open(OUTPUT_DIR / "feature_sets_hybrid_corr_stable_hie_alpha0p5.json", "w", encoding="utf-8") as f:
    json.dump(FEATURE_SETS, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / "feature_set_meta_hybrid_corr_stable_hie_alpha0p5.json", "w", encoding="utf-8") as f:
    json.dump(FEATURE_SET_META, f, ensure_ascii=False, indent=2)

feature_sets_summary = pd.DataFrame([
    {
        "set_name": name,
        "family": meta["family"],
        "z_window": meta["z_window"],
        "alpha_out": meta["alpha_out"],
        "n_features": meta["n_features"],
        "features": "|".join(meta["features"]),
    }
    for name, meta in FEATURE_SET_META.items()
])
feature_sets_summary.to_csv(OUTPUT_DIR / "feature_sets_summary_alpha0p5.csv", index=False)

print("Saved feature sets:", len(FEATURE_SETS))
display(feature_sets_summary)


Saved feature sets: 8


,set_name,family,z_window,alpha_out,n_features,features
0,z24_a0p5_corr_pruned_top10,corr_pruned,24,0.5,10,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...
1,z24_a0p5_hybrid_corr5_stablehie5_top10,hybrid_corr_stable_hie,24,0.5,10,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...
2,z48_a0p5_corr_pruned_top10,corr_pruned,48,0.5,10,range_sqrt_z|vol_ratio_72_log1p_z|MACD_hist_pc...
3,z48_a0p5_hybrid_corr5_stablehie5_top10,hybrid_corr_stable_hie,48,0.5,10,range_sqrt_z|vol_ratio_72_log1p_z|MACD_hist_pc...
4,z72_a0p5_corr_pruned_top10,corr_pruned,72,0.5,10,range_sqrt_z|vol_ratio_72_log1p_z|dist_to_high...
5,z72_a0p5_hybrid_corr5_stablehie5_top10,hybrid_corr_stable_hie,72,0.5,10,range_sqrt_z|vol_ratio_72_log1p_z|dist_to_high...
6,z96_a0p5_corr_pruned_top10,corr_pruned,96,0.5,10,range_sqrt_z|vol_ratio_24_log1p_z|vol_expansio...
7,z96_a0p5_hybrid_corr5_stablehie5_top10,hybrid_corr_stable_hie,96,0.5,10,range_sqrt_z|vol_ratio_24_log1p_z|vol_expansio...


## 11. Extra diagnostics: internal correlation and HIE attribution categories

In [15]:
# Internal correlation check for all saved sets.
internal_corr_rows = []
for set_name, features in FEATURE_SETS.items():
    meta = FEATURE_SET_META[set_name]
    z = int(meta["z_window"])
    train_df = FEATURE_TABLES_BY_Z[z]
    corr = train_df[features].corr(method="spearman").abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    max_corr = upper.max().max()
    max_pair = upper.stack().idxmax() if upper.stack().size and pd.notna(max_corr) else None
    internal_corr_rows.append({
        "set_name": set_name,
        "family": meta["family"],
        "z_window": z,
        "n_features": len(features),
        "max_abs_spearman_inside_set": float(max_corr) if pd.notna(max_corr) else np.nan,
        "max_corr_pair": str(max_pair),
        "passes_threshold": bool(pd.isna(max_corr) or max_corr < PRUNE_THRESHOLD),
    })

internal_corr_check = pd.DataFrame(internal_corr_rows)
internal_corr_check.to_csv(OUTPUT_DIR / "final_feature_set_internal_correlation_check.csv", index=False)
display(internal_corr_check)

# Attribution category summary.
category_summary = (
    main_hie_audit
    .groupby(["z_window", "hie_attribution_category"], as_index=False)
    .agg(
        n_features=("feature", "nunique"),
        n_selected_in_hybrid=("selected_in_hybrid", "sum"),
        median_corr_raw_rank=("corr_raw_rank", "median"),
        median_stability_count=("stability_count", "median"),
    )
)
category_summary.to_csv(OUTPUT_DIR / "hie_attribution_category_summary.csv", index=False)
display(category_summary)


,set_name,family,z_window,n_features,max_abs_spearman_inside_set,max_corr_pair,passes_threshold
0,z24_a0p5_corr_pruned_top10,corr_pruned,24,10,0.773066,"('vol_ratio_168_log1p_z', 'range_sqrt_z')",True
1,z24_a0p5_hybrid_corr5_stablehie5_top10,hybrid_corr_stable_hie,24,10,0.784903,"('price_pos_in_range_72_norm', 'RSI_72_bounded')",True
2,z48_a0p5_corr_pruned_top10,corr_pruned,48,10,0.847140,"('MOM_pct_14_z', 'ema_spread_9_21_z')",True
3,z48_a0p5_hybrid_corr5_stablehie5_top10,hybrid_corr_stable_hie,48,10,0.784533,"('MACD_hist_pct_z', 'MOM_pct_14_z')",True
4,z72_a0p5_corr_pruned_top10,corr_pruned,72,10,0.799465,"('NATR_14_z', 'volatility_24_sqrt_z')",True
5,z72_a0p5_hybrid_corr5_stablehie5_top10,hybrid_corr_stable_hie,72,10,0.776829,"('log_ret_72_z', 'ema_slope_100_z')",True
6,z96_a0p5_corr_pruned_top10,corr_pruned,96,10,0.845165,"('range_sqrt_z', 'vol_expansion_24_log1p_z')",True
7,z96_a0p5_hybrid_corr5_stablehie5_top10,hybrid_corr_stable_hie,96,10,0.845165,"('range_sqrt_z', 'vol_expansion_24_log1p_z')",True


,z_window,hie_attribution_category,n_features,n_selected_in_hybrid,median_corr_raw_rank,median_stability_count
0,24,corr_ranked_but_removed_by_pruning,2,0,40.5,2.0
1,24,not_in_corr_pruned_raw_rank_available,7,5,37.0,2.0
2,48,corr_ranked_but_removed_by_pruning,1,0,41.0,2.0
3,48,not_in_corr_pruned_raw_rank_available,10,5,31.5,2.0
4,72,corr_ranked_but_removed_by_pruning,1,1,23.0,2.0
5,72,not_in_corr_pruned_raw_rank_available,8,4,29.5,2.0
6,72,shared_with_corr_pruned,2,1,10.0,2.0
7,96,corr_ranked_but_removed_by_pruning,1,0,28.0,3.0
8,96,not_in_corr_pruned_raw_rank_available,8,5,25.0,2.0
9,96,shared_with_corr_pruned,1,0,10.0,2.0


## 12. Rebuild hybrid sizes without recomputing HIE

Этот блок можно запускать сколько угодно раз: он использует сохранённые в памяти `hie_topk_long`, `stable_hie_per_z_main`, `CORR_RANKINGS_BY_Z`, `FEATURE_TABLES_BY_Z`.

Примеры:
- 10 признаков = 5 corr + до 5 HIE.
- 13 признаков = 6 corr + до 7 HIE.
- 14 признаков = 7 corr + до 7 HIE.


In [ ]:
HYBRID_SIZE_SCENARIOS = [
    {
        "scenario_name": "hybrid_top10_corr5_hie5",
        "final_k": 10,
        "corr_core_k": 5,
        "hie_target_k": 5,
    },
    {
        "scenario_name": "hybrid_top13_corr6_hie7",
        "final_k": 13,
        "corr_core_k": 6,
        "hie_target_k": 7,
    },
    {
        "scenario_name": "hybrid_top14_corr7_hie7",
        "final_k": 14,
        "corr_core_k": 7,
        "hie_target_k": 7,
    },
]

SCENARIO_OUTPUTS = {}
all_scenario_summaries = []
all_scenario_audits = []
all_scenario_steps = []

for sc in HYBRID_SIZE_SCENARIOS:
    name = sc["scenario_name"]
    print("Building scenario:", name)
    results, summary, steps, audit = build_hybrid_all_z_from_cached(
        scenario_name=name,
        final_k=sc["final_k"],
        corr_core_k=sc["corr_core_k"],
        hie_target_k=sc["hie_target_k"],
        stability_top_k=MAIN_HIE_STABILITY_TOPK,
        min_stability_count=HIE_MIN_STABILITY_COUNT,
        prune_threshold=PRUNE_THRESHOLD,
        strict_exclude_raw_corr_top_n_for_incremental=STRICT_EXCLUDE_RAW_CORR_TOP_N_FOR_INCREMENTAL,
    )
    SCENARIO_OUTPUTS[name] = {
        "results": results,
        "summary": summary,
        "steps": steps,
        "audit": audit,
    }
    all_scenario_summaries.append(summary)
    all_scenario_steps.append(steps)
    all_scenario_audits.append(audit)

scenario_summaries_all = pd.concat(all_scenario_summaries, ignore_index=True)
scenario_steps_all = pd.concat(all_scenario_steps, ignore_index=True)
scenario_audits_all = pd.concat(all_scenario_audits, ignore_index=True)

scenario_summaries_all.to_csv(OUTPUT_DIR / "all_hybrid_size_scenarios_summary.csv", index=False)
scenario_steps_all.to_csv(OUTPUT_DIR / "all_hybrid_size_scenarios_selection_steps.csv", index=False)
scenario_audits_all.to_csv(OUTPUT_DIR / "all_hybrid_size_scenarios_hie_vs_corr_audit.csv", index=False)

display(scenario_summaries_all)


## 13. Optional: create feature_sets.json for a selected scenario

Измени `SELECTED_SCENARIO_FOR_EXPORT`, если хочешь экспортировать top13/top14 вместо top10.


In [ ]:
SELECTED_SCENARIO_FOR_EXPORT = "hybrid_top10_corr5_hie5"
# SELECTED_SCENARIO_FOR_EXPORT = "hybrid_top13_corr6_hie7"
# SELECTED_SCENARIO_FOR_EXPORT = "hybrid_top14_corr7_hie7"

if SELECTED_SCENARIO_FOR_EXPORT not in SCENARIO_OUTPUTS:
    raise KeyError(f"Unknown scenario: {SELECTED_SCENARIO_FOR_EXPORT}")

scenario_results = SCENARIO_OUTPUTS[SELECTED_SCENARIO_FOR_EXPORT]["results"]

feature_sets_scenario = {}
feature_set_meta_scenario = {}

for r in scenario_results:
    z = int(r["z_window"])
    alpha_tag = alpha_to_tag(MAIN_HIE_ALPHA_OUT)

    corr_name = f"z{z}_{alpha_tag}_corr_pruned_top{r['final_k']}"
    hybrid_name = f"z{z}_{alpha_tag}_{SELECTED_SCENARIO_FOR_EXPORT}"

    feature_sets_scenario[corr_name] = r["corr_pruned"]
    feature_sets_scenario[hybrid_name] = r["hybrid"]

    feature_set_meta_scenario[corr_name] = {
        "set_name": corr_name,
        "family": "corr_pruned",
        "z_window": z,
        "alpha_out": float(MAIN_HIE_ALPHA_OUT),
        "n_features": len(r["corr_pruned"]),
        "features": r["corr_pruned"],
    }
    feature_set_meta_scenario[hybrid_name] = {
        "set_name": hybrid_name,
        "family": "hybrid_corr_stable_hie",
        "z_window": z,
        "alpha_out": float(MAIN_HIE_ALPHA_OUT),
        "n_features": len(r["hybrid"]),
        "features": r["hybrid"],
        "corr_pruned_reference": r["corr_pruned"],
        "jaccard_with_corr": r["jaccard"],
        "n_hie_incremental_kept": r["n_hie_incremental_kept"],
        "n_corr_core_kept": r["n_corr_core_kept"],
        "n_corr_fallback_kept": r["n_corr_fallback_kept"],
        "n_hie_fallback_kept": r["n_hie_fallback_kept"],
    }

export_dir = OUTPUT_DIR / SELECTED_SCENARIO_FOR_EXPORT
export_dir.mkdir(parents=True, exist_ok=True)

with open(export_dir / "feature_sets.json", "w", encoding="utf-8") as f:
    json.dump(feature_sets_scenario, f, ensure_ascii=False, indent=2)

with open(export_dir / "feature_set_meta.json", "w", encoding="utf-8") as f:
    json.dump(feature_set_meta_scenario, f, ensure_ascii=False, indent=2)

pd.DataFrame([
    {
        "set_name": name,
        "family": meta["family"],
        "z_window": meta["z_window"],
        "alpha_out": meta["alpha_out"],
        "n_features": meta["n_features"],
        "features": "|".join(meta["features"]),
    }
    for name, meta in feature_set_meta_scenario.items()
]).to_csv(export_dir / "feature_sets_table.csv", index=False)

print("Exported scenario to:", export_dir)
display(pd.read_csv(export_dir / "feature_sets_table.csv"))


## 14. Auto-summary for defense

In [ ]:
summary_lines = []
summary_lines.append("Hybrid HIE-stability + correlation feature selection summary")
summary_lines.append("=" * 72)
summary_lines.append(f"Main HIE alpha_out: {MAIN_HIE_ALPHA_OUT}")
summary_lines.append(f"Main HIE stability top-K: {MAIN_HIE_STABILITY_TOPK}")
summary_lines.append(f"HIE stability sensitivity grid: {HIE_TOPK_STABILITY_GRID}")
summary_lines.append(f"Stable criterion: appears in at least {HIE_MIN_STABILITY_COUNT}/3 train blocks")
summary_lines.append(f"Pruning threshold: Spearman abs < {PRUNE_THRESHOLD}")
summary_lines.append("")
summary_lines.append("Stability sensitivity by top-K:")
if 'stability_robustness_df' in globals() and not stability_robustness_df.empty:
    for _, row in stability_robustness_df.iterrows():
        summary_lines.append(
            f"  z={int(row['z_window'])}, K={int(row['top_k_stability'])}: "
            f"stable>=2={int(row['n_stable_ge_2'])}, "
            f"count3={int(row['n_count_3'])}, count2={int(row['n_count_2'])}"
        )
summary_lines.append("")
summary_lines.append("Main scenario hybrid vs corrPrunedTop10:")
for _, row in main_summary.iterrows():
    summary_lines.append(
        f"  z={int(row['z_window'])}: Jaccard={row['jaccard']:.3f}, "
        f"overlap={int(row['overlap_count'])}/{int(row['final_k'])}, "
        f"HIE kept={int(row['n_hie_incremental_kept'])}, "
        f"corr_core kept={int(row['n_corr_core_kept'])}"
    )

summary_lines.append("")
summary_lines.append("Alpha overlap demo alpha=0.5 vs alpha=1.0:")
if 'alpha_overlap_df' in globals() and not alpha_overlap_df.empty:
    for _, row in alpha_overlap_df.iterrows():
        summary_lines.append(
            f"  z={int(row['z_window'])}, {row['comparison']}, top{int(row['top_k'])}: "
            f"Jaccard={row['jaccard']:.3f}, overlap={int(row['overlap_count'])}/{int(row['top_k'])}"
        )

summary_lines.append("")
summary_lines.append("Defense wording:")
summary_lines.append(
    "HIE stability is measured on unpruned entry/exit union rankings across chronological train blocks. The main top-K is 14, tied to the 2x buffer around the intended 6-7 HIE quota; K={7,14,20,26} is reported as sensitivity because stable sets grow monotonically with K. "
    "Spearman pruning is not used during stability counting to avoid artificial instability from correlated feature-family swaps. "
    "Pruning is applied only when constructing the final corr and hybrid feature sets for the linear bandit. "
    "For every stable HIE candidate, raw Spearman-correlation rank is audited to distinguish genuinely HIE-distinct features "
    "from correlation-accessible features removed by pruning."
)

summary_text = "\n".join(summary_lines)
with open(OUTPUT_DIR / "defense_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)

print(summary_text)
